Baseline data extraction, delay calculation, fairness setup, and capacity penalty construction

This block prepares the full “before control” baseline using the PeMS 5-minute dataset for the
selected I-405 southbound corridor. The goal is to extract the observed freeway and ramp data,
compute benchmark mainline delay, build local-road delay scenarios, define the fairness penalty,
and construct the three capacity-related penalty terms that will later be reused inside the CTM
and ADMM model.

--------------------------------------------------------------------------------------------------
Part 1. Load the raw PeMS dataset and extract only the corridor stations we care about

The code first loads:

* the station metadata file, which helps verify detector IDs and locations, and
* the 5-minute station dataset, which contains timestamped flow, occupancy, and speed values.

Then it filters the large district-wide dataset so that only the selected station IDs remain.
These IDs include:

* mainline detectors
* on-ramp detectors
* off-ramp detectors

Two time windows are extracted:

1. a low-congestion / early-morning window, used to estimate free-flow speed
2. the 08:00 morning-congestion window, used to measure observed corridor conditions

After filtering, only the important columns are kept:

* timestamp
* station_id
* station_type
* total_flow
* avg_occupancy
* avg_speed

This reduced dataset becomes the starting point for all later calculations.

--------------------------------------------------------------------------------------------------
Part 2. Mainline delay benchmark using observed PeMS data

This section computes a flow-based mainline-delay baseline, meaning it uses observed flow and
observed travel time directly from the dataset, before any CTM or ADMM control is applied.

The computation is done in several steps.

Step 1. Estimate free-flow speed for each mainline station

For each selected mainline detector, the code computes the median speed during the chosen
low-congestion time window.

This gives a station-level free-flow speed estimate:

$$\tilde v_s=\text{median of overnight speed samples at station } s$$

The reason for using the median is to reduce the effect of occasional noisy speed values.

Step 2. Convert station free-flow speeds into segment free-flow speeds

Each corridor segment lies between two consecutive mainline stations.
The free-flow speed of a segment is taken as the average of the upstream and downstream station
free-flow speeds:

$$v_i^{ff}=\frac{\tilde v_{up}+\tilde v_{down}}{2}$$

Step 3. Compute free-flow travel time for each segment

Once free-flow speed is known, free-flow travel time is:

$$TT_i^{ff}=\frac{L_i}{v_i^{ff}}$$

where:

* $L_i$ = segment length in miles
* $v_i^{ff}$ = free-flow speed in mph

The code stores both:

* travel time in hours
* travel time in minutes

Step 4. Compute observed segment speed at 08:00

For the morning peak dataset, the code gets the actual observed speed at each mainline station.
Then, for each segment, it averages the upstream and downstream observed speeds:

$$v_i^{obs}=\frac{v_{up}^{obs}+v_{down}^{obs}}{2}$$

Step 5. Compute observed travel time for each segment

Using the observed segment speed:

$$TT_i^{obs}=\frac{L_i}{v_i^{obs}}$$

This gives the actual travel time experienced during the selected congested 5-minute interval.

Step 6. Compute delay per vehicle

The segment delay per vehicle is the difference between observed travel time and free-flow travel time:

$$d_i=TT_i^{obs}-TT_i^{ff}$$

This is computed in both hours and minutes.

Step 7. Compute total mainline delay for each segment

The code then multiplies per-vehicle delay by the observed segment flow:

$$D_{M,i}=q_i \cdot d_i$$

where:

* $q_i$ = observed segment flow during the 5-minute interval
* $d_i$ = per-vehicle delay

This gives total mainline delay in:

* vehicle-minutes
* vehicle-hours

This result is the observed, no-control benchmark for the freeway mainline.

NOTE:
This is a flow-based benchmark. Later, the CTM model will use a state-based mainline delay
formulation, which includes stored vehicles inside the cells and is better suited for dynamic
simulation and optimization.

--------------------------------------------------------------------------------------------------
Part 3. Local-road delay setup using ramp queue scenarios

The raw dataset gives observed ramp discharge flow, but it does not directly provide the full
ramp queue state at the beginning of the interval. Because of that, this section uses scenario-
based assumptions to understand how local-road delay could behave under different queue and
arrival conditions.

Step 1. Extract 08:00 on-ramp flow values

The code filters the dataset to keep only the on-ramp detectors at the selected time.
These ramp flows are used as:

* discharged ramp flow, $u_t$

Step 2. Define the ramp queue update equation

Ramp state evolves using:

$$R_t = R_{t-1}+a_t-u_t$$

where:

* $R_{t-1}$ = queue at the beginning of the interval
* $a_t$ = newly arriving vehicles to the ramp during the interval
* $u_t$ = discharged vehicles released from the ramp

Step 3. Define local-road delay

Local delay is measured using the trapezoidal approximation:

$$D_{L,i}(t)=\left(\frac{R_{t-1}+R_t}{2}\right)\Delta T$$

where:

* $\Delta T = 5$ minutes in this baseline calculation block

Step 4. Test multiple queue scenarios

Because the initial queue is unknown, the code tests several possible values:

* $R_{t-1}\in \{0,10,20,30\}$

Step 5. Test multiple arrival assumptions

Because actual ramp arrivals are unknown, the code tests:

* $a_t = u_t$
* $a_t = 1.5u_t$
* $a_t = 2u_t$

For every combination of:

* initial queue assumption
* arrival assumption

the code computes:

* next ramp queue
* local delay for each ramp
* total local delay across all ramps

This gives a scenario-based local-road benchmark and shows how sensitive local delay is to the
assumed arrival and initial queue values.

--------------------------------------------------------------------------------------------------
Part 4. Fairness penalty setup using ramp stress

The purpose of the fairness term is to discourage a control policy from overloading one ramp much
more than the others.

Step 1. Compute maximum ramp storage

Ramp maximum queue storage is estimated from physical ramp length and number of lanes:

$$R_{max}=\frac{\text{ramp length}\times \text{number of lanes}}{\text{average vehicle length}}$$

where:

* average vehicle length is assumed to be 25 ft

This gives the approximate maximum number of vehicles each ramp can physically store.

Step 2. Compute ramp stress index

Ramp stress is defined as:

$$\phi_i=\frac{R_i}{R_{max,i}}$$

Interpretation:

* $\phi_i=0$ means empty ramp
* $\phi_i=0.5$ means ramp is half full
* $\phi_i=1$ means ramp is full
* $\phi_i>1$ means spillback is occurring

Step 3. Cap stress at 1 for fairness

Since fairness is meant to compare relative burden rather than reward extreme overflow, stress is
capped:

$$\phi_i^{cap}=\min(\phi_i,1)$$

This means any ramp already beyond full storage is treated as fully stressed for fairness
comparison.

Step 4. Compute pairwise fairness penalty

The fairness penalty compares every pair of ramps:

$$L_{fair}=\gamma \sum_{i<j}(\phi_i^{cap}-\phi_j^{cap})^2$$

where:

* $\gamma$ = fairness multiplier

The code evaluates this penalty for:

* several ramp queue assumptions
* several arrival assumptions
* several fairness multipliers

This shows how fairness changes when ramp congestion becomes more uneven.

Interpretation:

* equal stress across ramps -> low fairness penalty
* one or more ramps much more stressed than others -> high fairness penalty

--------------------------------------------------------------------------------------------------
Part 5. Doorway capacity construction

This section builds the first capacity-related term: doorway capacity.

Doorway capacity represents the maximum number of vehicles that can enter / pass through a segment
during one interval before flow-processing limits are exceeded.

Step 1. Start from known station capacities

Some stations already have known doorway capacities in vehicles per 5-minute interval.

Step 2. Convert known capacities into per-lane values

For each station with known capacity:

$$\text{per-lane doorway capacity}=\frac{\text{known total capacity}}{\text{number of lanes}}$$

Step 3. Average across stations

The code then computes an average per-lane doorway capacity using the known stations.

Step 4. Estimate missing capacities

For stations with missing doorway capacity, the code estimates:

$$C_i=(\text{average per-lane doorway capacity})\times(\text{lane count})$$

This produces a full doorway-capacity table for all corridor stations / segments.

--------------------------------------------------------------------------------------------------
Part 6. Doorway capacity penalty

The doorway capacity penalty activates when total inflow into a segment exceeds its doorway
capacity.

Formula:

$$L_{\text{doorway},i}=\lambda_1 \max(0,Q_{in,i}+u_i-C_i)^2$$

where:

* $Q_{in,i}$ = mainline inflow to segment i
* $u_i$ = total on-ramp inflow entering segment i
* $C_i$ = doorway capacity of segment i
* $\lambda_1$ = doorway penalty multiplier

The code tests this penalty using:

* actual observed flow values
* synthetic larger flow values

The synthetic case is used as a sanity check to make sure the penalty activates correctly when
inflow exceeds capacity.

--------------------------------------------------------------------------------------------------
Part 7. Physical storage capacity

Physical capacity is the hard upper limit on how many vehicles can physically fit inside a
segment.

Formula:

$$N_{max,i}=k_j \cdot L_i \cdot n_i$$

where:

* $k_j$ = jam density in veh/mi/ln
* $L_i$ = segment length
* $n_i$ = number of lanes

This gives the maximum segment storage before full jam density is reached.

--------------------------------------------------------------------------------------------------
Part 8. Initial and final mainline state for the segment-based baseline

To evaluate storage-related penalties, the code needs an estimate of how many vehicles are already
inside each segment at the beginning and end of the 5-minute interval.

Step 1. Compute station density

Station density is estimated using observed flow and speed:

$$k=\frac{q^{hr}}{v}$$

Since PeMS flow is reported per 5 minutes, it is converted to hourly flow first:

$$q^{hr}=12\cdot q^{5min}$$

Step 2. Compute average segment density

For each segment, density is averaged between the upstream and downstream stations.

Step 3. Compute initial vehicles in segment

$$X_{initial,i}=k_{avg,i}\cdot L_i$$

Step 4. Compute final vehicles in segment using conservation

$$X_{final,i}=X_{initial,i}+Q_{in,i}+u_i-Q_{out,i}-f_i$$

where:

* $Q_{in,i}$ = mainline inflow
* $u_i$ = on-ramp inflow
* $Q_{out,i}$ = mainline outflow
* $f_i$ = off-ramp outflow

This gives an approximate segment-level state update using the observed 5-minute data.

--------------------------------------------------------------------------------------------------
Part 9. Physical-capacity penalty

This penalty activates only when the final segment occupancy exceeds hard physical storage.

Formula:

$$L_{\text{physical},i}=\lambda_3 \max(0,X_{final,i}-N_{max,i})^2$$

where:

* $X_{final,i}$ = estimated vehicles in segment i at the end of the interval
* $N_{max,i}$ = physical storage limit
* $\lambda_3$ = physical-capacity multiplier

The code first checks this using actual data.
Since actual occupancies are below hard physical storage, the penalty becomes zero.

Then the code uses synthetic oversized states as a sanity check to verify that the penalty logic
works correctly when overflow exists.

--------------------------------------------------------------------------------------------------
Part 10. Safe-threshold capacity

Safe threshold is a soft storage limit placed below the hard physical maximum.
It is used to penalize unstable congestion before the segment reaches full jam density.

Formula:

$$X_{safe,i}=\eta N_{max,i}$$

where:

* $\eta$ = threshold multiplier, usually between 0 and 1
* $N_{max,i}$ = physical capacity

The code tests multiple values of:

* $\eta = 0.3, 0.5, 0.7$

This allows inspection of how aggressive or relaxed the soft-threshold should be.

--------------------------------------------------------------------------------------------------
Part 11. Safe-threshold penalty

This penalty activates when segment occupancy exceeds the soft threshold:

$$L_{\text{safe},i}=\lambda_2 \max(0,X_{final,i}-X_{safe,i})^2$$

where:

* $X_{final,i}$ = estimated final segment occupancy
* $X_{safe,i}$ = soft threshold
* $\lambda_2$ = safe-threshold multiplier

The code evaluates this penalty for:

* actual segment states
* synthetic oversized states
* several choices of eta
* several values of lambda_2

This shows:

* whether the actual data crosses the soft threshold
* how sensitive the penalty is to the threshold choice
* how strongly the penalty scales with lambda_2

--------------------------------------------------------------------------------------------------
Final interpretation of this whole block

This code block builds the full baseline and penalty structure needed before moving to CTM and
ADMM.

In summary, it does five major jobs:

1. extracts observed mainline, ramp, and off-ramp data from PeMS
2. computes the observed no-control mainline delay benchmark
3. explores local-road delay and fairness using ramp queue scenarios
4. constructs the three mainline capacity terms:
   * doorway capacity
   * safe threshold
   * physical capacity
5. verifies all penalty equations using both actual and synthetic test values

This section is important because it creates the numerical baseline, penalty definitions, and
calibration logic that are later reused when the corridor is rebuilt as a CTM network and
optimized with ADMM-based ramp control.

In [48]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
%matplotlib inline


In [49]:
# 1. Read station metadata and display selected stations clearly
import pandas as pd

# Final selected detector IDs:
# 7 mainline + 5 on-ramp + 3 off-ramp
ids_to_keep = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620,
    1201460, 1201490, 1201517, 1201548, 1201580,
    1201465, 1201554, 1201585
]

ids_to_keep_str = {str(x).strip() for x in ids_to_keep}




ids_to_keep_str = {str(x).strip() for x in ids_to_keep}

metadata_raw = pd.read_csv(
    "Station Metadata_district12.txt",
    sep=None,
    engine="python",
    header=None,
    dtype=str,
    skip_blank_lines=True
)

metadata_raw = metadata_raw.apply(lambda col: col.astype(str).str.strip())

selected_metadata_clean = pd.DataFrame({
    "station_id": metadata_raw[0],
    "station_name": metadata_raw[13],
    "station_type": metadata_raw[11],
    "freeway": metadata_raw[1],
    "direction": metadata_raw[2],
    "absolute_postmile": metadata_raw[7],
    "length": metadata_raw[10],
    "lanes": metadata_raw[12],
    "latitude": metadata_raw[8],
    "longitude": metadata_raw[9],
})

# Keep only selected stations
selected_metadata_clean["station_id"] = selected_metadata_clean["station_id"].astype(str).str.strip()

selected_metadata_clean = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(ids_to_keep_str)
].copy()

# Convert numeric columns
numeric_cols = [
    "station_id",
    "freeway",
    "absolute_postmile",
    "length",
    "lanes",
    "latitude",
    "longitude",
]

for col in numeric_cols:
    selected_metadata_clean[col] = pd.to_numeric(selected_metadata_clean[col], errors="coerce")

# Sort by corridor position
selected_metadata_clean = selected_metadata_clean.sort_values(
    ["absolute_postmile", "station_type"]
).reset_index(drop=True)

print("=== Selected Station Metadata ===")
print("Number of selected stations:", len(selected_metadata_clean))

display(selected_metadata_clean)

=== Selected Station Metadata ===
Number of selected stations: 15


,station_id,station_name,station_type,freeway,direction,absolute_postmile,length,lanes,latitude,longitude
0,1201419,RED HILL,ML,405,S,8.17,0.269,5,33.686517,-117.866474
1,1201465,BRISTOL 1,FR,405,S,9.31,NaN,2,33.687251,-117.886180
2,1201469,BRISTOL 1,ML,405,S,9.31,0.350,5,33.687251,-117.886180
3,1201460,BRISTOL 1,OR,405,S,9.31,NaN,1,33.687251,-117.886180
4,1201497,FAIRVIEW,ML,405,S,10.05,0.460,5,33.687480,-117.899035
5,1201490,FAIRVIEW,OR,405,S,10.07,NaN,1,33.687494,-117.899383
6,1201525,HARBOR 1,ML,405,S,10.97,0.610,6,33.687942,-117.915002
7,1201517,HARBOR 1,OR,405,S,10.97,NaN,1,33.687942,-117.915002
8,1201554,HARBOR 2,FR,405,S,11.27,NaN,3,33.689212,-117.919950
9,1201558,HARBOR 2,ML,405,S,11.27,0.480,5,33.689212,-117.919950


In [50]:

# 2. Load PeMS 5-minute station data
# Purpose:
# Load the full 5-minute detector dataset and create the 01:00–05:00
# free-flow window used to estimate free-flow speed.

# Load 5-minute PeMS station data
station_data = pd.read_csv("station_5min_district12.txt", header=None)

# Column 0 = timestamp
station_data[0] = pd.to_datetime(station_data[0])

# Final selected detector IDs:
# 7 mainline + 5 on-ramp + 3 off-ramp
ids_to_keep = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620,
    1201460, 1201490, 1201517, 1201548, 1201580,
    1201465, 1201554, 1201585
]

# Free-flow window
# This window is used only to estimate free-flow speed from low-congestion conditions.
start_time = "2026-01-08 01:00:00"
end_time   = "2026-01-08 05:00:00"

# Filter selected detectors during free-flow window
filtered_data_midnight = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

# Keep only useful columns
selected_data_midnight = filtered_data_midnight[[0, 1, 5, 9, 10, 11]].copy()

selected_data_midnight.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

# Sort for readability
selected_data_midnight = selected_data_midnight.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print("Midnight / Free-Flow Window Data ")
print("Time window: 2026-01-08 01:00:00 to 2026-01-08 05:00:00")
print("Rows:", len(selected_data_midnight))
print("Unique timestamps:", selected_data_midnight["timestamp"].nunique())
print("Unique stations:", selected_data_midnight["station_id"].nunique())

print("\nFirst 20 rows:")
display(selected_data_midnight.head(20))

print("\nRows for station 1201419 (RED HILL):")
display(selected_data_midnight[selected_data_midnight["station_id"] == 1201419].head(20))

Midnight / Free-Flow Window Data 
Time window: 2026-01-08 01:00:00 to 2026-01-08 05:00:00
Rows: 720
Unique timestamps: 48
Unique stations: 15

First 20 rows:


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 01:00:00,1201419,ML,112.0,0.0345,47.3
1,2026-01-08 01:00:00,1201460,OR,0.0,0.0000,NaN
2,2026-01-08 01:00:00,1201465,FR,NaN,NaN,NaN
3,2026-01-08 01:00:00,1201469,ML,112.0,0.0257,61.4
4,2026-01-08 01:00:00,1201490,OR,9.0,0.0200,NaN
5,2026-01-08 01:00:00,1201497,ML,55.0,0.0049,69.8
6,2026-01-08 01:00:00,1201517,OR,3.0,0.0030,NaN
7,2026-01-08 01:00:00,1201525,ML,54.0,0.0097,64.5
8,2026-01-08 01:00:00,1201548,OR,6.0,0.0110,NaN
9,2026-01-08 01:00:00,1201554,FR,10.0,0.0033,NaN



Rows for station 1201419 (RED HILL):


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 01:00:00,1201419,ML,112.0,0.0345,47.3
15,2026-01-08 01:05:00,1201419,ML,174.0,0.0332,53.4
30,2026-01-08 01:10:00,1201419,ML,177.0,0.0398,53.7
45,2026-01-08 01:15:00,1201419,ML,217.0,0.0335,64.4
60,2026-01-08 01:20:00,1201419,ML,206.0,0.0344,68.4
75,2026-01-08 01:25:00,1201419,ML,194.0,0.0342,68.2
90,2026-01-08 01:30:00,1201419,ML,179.0,0.0312,68.0
105,2026-01-08 01:35:00,1201419,ML,155.0,0.0361,60.9
120,2026-01-08 01:40:00,1201419,ML,184.0,0.0401,57.7
135,2026-01-08 01:45:00,1201419,ML,188.0,0.0335,63.1


In [51]:
# 3. Load 08:00–09:00 benchmark window data
# Purpose:
# This is the actual peak-hour window used for the 8-cell / 30-sec CTM benchmark.
# PeMS gives 5-minute data, so 08:00–09:00 gives 12 intervals:
# 08:00, 08:05, ..., 08:55
# Each 5-min interval will later be divided into ten 30-sec CTM steps.

start_time = "2026-01-08 08:00:00"
end_time   = "2026-01-08 09:00:00"

filtered_data_morning = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

selected_data_morning = filtered_data_morning[[0, 1, 5, 9, 10, 11]].copy()

selected_data_morning.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

selected_data_morning = selected_data_morning.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print(" Morning Benchmark Window Data ")
print("Time window: 2026-01-08 08:00:00 to 2026-01-08 09:00:00")
print("Rows:", len(selected_data_morning))
print("Unique timestamps:", selected_data_morning["timestamp"].nunique())
print("Unique stations:", selected_data_morning["station_id"].nunique())

print("\nFirst 30 rows:")
display(selected_data_morning.head(30))

print("\nRows for station 1201419 (RED HILL)")
display(selected_data_morning[selected_data_morning["station_id"] == 1201419])

 Morning Benchmark Window Data 
Time window: 2026-01-08 08:00:00 to 2026-01-08 09:00:00
Rows: 180
Unique timestamps: 12
Unique stations: 15

First 30 rows:


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 08:00:00,1201419,ML,645.0,0.1582,39.4
1,2026-01-08 08:00:00,1201460,OR,88.0,0.0840,NaN
2,2026-01-08 08:00:00,1201465,FR,NaN,NaN,NaN
3,2026-01-08 08:00:00,1201469,ML,681.0,0.1384,40.0
4,2026-01-08 08:00:00,1201490,OR,39.0,0.8700,NaN
5,2026-01-08 08:00:00,1201497,ML,622.0,0.0525,69.5
6,2026-01-08 08:00:00,1201517,OR,73.0,0.0980,NaN
7,2026-01-08 08:00:00,1201525,ML,737.0,0.1677,30.6
8,2026-01-08 08:00:00,1201548,OR,56.0,0.0980,NaN
9,2026-01-08 08:00:00,1201554,FR,82.0,0.0283,NaN



Rows for station 1201419 (RED HILL)


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 08:00:00,1201419,ML,645.0,0.1582,39.4
15,2026-01-08 08:05:00,1201419,ML,677.0,0.1536,39.8
30,2026-01-08 08:10:00,1201419,ML,703.0,0.1708,39.6
45,2026-01-08 08:15:00,1201419,ML,769.0,0.1629,43.6
60,2026-01-08 08:20:00,1201419,ML,726.0,0.1420,44.4
75,2026-01-08 08:25:00,1201419,ML,757.0,0.1616,47.5
90,2026-01-08 08:30:00,1201419,ML,772.0,0.1622,46.8
105,2026-01-08 08:35:00,1201419,ML,670.0,0.1642,44.0
120,2026-01-08 08:40:00,1201419,ML,715.0,0.1535,43.7
135,2026-01-08 08:45:00,1201419,ML,587.0,0.1551,38.0


In [52]:
# 5. Column reference for PeMS station_5min data

pems_column_reference = pd.DataFrame({
    "column_index": [0, 1, 5, 9, 10, 11],
    "column_name": [
        "timestamp",
        "station_id",
        "station_type",
        "total_flow",
        "avg_occupancy",
        "avg_speed"
    ]

})

display(pems_column_reference)

,column_index,column_name
0,0,timestamp
1,1,station_id
2,5,station_type
3,9,total_flow
4,10,avg_occupancy
5,11,avg_speed


## below is the starting point of the calculations using the dataset we collected


# mainline delay calculations


Free-flow speeds were estimated from the 01:00–05:00 low-congestion window. For each mainline detector, all 48 five-minute speed observations were collected, and the median speed was used as the station-level free-flow speed. The median was used instead of the mean to reduce sensitivity to abnormal detector readings or short disturbances.

In [53]:

# 5. Estimate free-flow speed for each mainline station
# Purpose:
# Use the low-congestion midnight window, 01:00–05:00,
# to estimate free-flow speed at each mainline detector.


mainline_ids = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620
]

mainline_midnight_data = selected_data_midnight[
    (selected_data_midnight["station_id"].isin(mainline_ids)) &
    (selected_data_midnight["station_type"] == "ML")
].copy()

mainline_midnight_data["avg_speed"] = pd.to_numeric(
    mainline_midnight_data["avg_speed"],
    errors="coerce"
)

median_speed_by_station = (
    mainline_midnight_data
    .groupby("station_id", as_index=False)["avg_speed"]
    .median()
)

# IMPORTANT:
# Keep this column name as "median_speed"
# because the next block uses median_speed_df["median_speed"]
median_speed_by_station.columns = ["station_id", "median_speed"]

# Optional clean display table, but do NOT change median_speed_by_station
median_speed_display = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(mainline_ids)
][
    ["station_id", "station_name", "absolute_postmile"]
].merge(
    median_speed_by_station,
    on="station_id",
    how="left"
).sort_values("absolute_postmile").reset_index(drop=True)

print("=== Median Free-Flow Speed by Mainline Station ===")
display(median_speed_display)

=== Median Free-Flow Speed by Mainline Station ===


,station_id,station_name,absolute_postmile,median_speed
0,1201419,RED HILL,8.17,64.20
1,1201469,BRISTOL 1,9.31,65.35
2,1201497,FAIRVIEW,10.05,69.85
3,1201525,HARBOR 1,10.97,68.40
4,1201558,HARBOR 2,11.27,68.70
5,1201589,EUCLID,12.27,68.05
6,1201620,TALBERT,13.07,67.95


In [54]:

# Verify median free-flow speed calculation
mainline_midnight_check = selected_data_midnight[
    (selected_data_midnight["station_id"].isin(mainline_ids)) &
    (selected_data_midnight["station_type"] == "ML")
].copy()

mainline_midnight_check["avg_speed"] = pd.to_numeric(
    mainline_midnight_check["avg_speed"],
    errors="coerce"
)

speed_check = (
    mainline_midnight_check
    .groupby("station_id")["avg_speed"]
    .agg(
        count="count",
        min_speed="min",
        median_speed="median",
        max_speed="max"
    )
    .reset_index()
)

speed_check = speed_check.merge(
    selected_metadata_clean[["station_id", "station_name"]],
    on="station_id",
    how="left"
)

speed_check = speed_check[
    ["station_id", "station_name", "count", "min_speed", "median_speed", "max_speed"]
]

display(speed_check)

,station_id,station_name,count,min_speed,median_speed,max_speed
0,1201419,RED HILL,48,47.3,64.20,78.7
1,1201469,BRISTOL 1,48,49.0,65.35,69.4
2,1201497,FAIRVIEW,48,39.7,69.85,74.4
3,1201525,HARBOR 1,48,63.4,68.40,72.9
4,1201558,HARBOR 2,48,64.9,68.70,73.8
5,1201589,EUCLID,48,63.4,68.05,74.3
6,1201620,TALBERT,48,62.5,67.95,73.9


In [55]:
# 6. Segment free-flow speed calculation
# Purpose:
# Use the median free-flow speed at each pair of neighboring
# mainline stations to estimate segment free-flow speed.

# segment free flow speed
def segment_free_flow_speed(upstream_id, downstream_id, median_speed_df):
    v_upstream = median_speed_df.loc[
        median_speed_df["station_id"] == upstream_id,
        "median_speed"
    ].iloc[0]

    v_downstream = median_speed_df.loc[
        median_speed_df["station_id"] == downstream_id,
        "median_speed"
    ].iloc[0]

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)

# free flow speed between each segment
v_ff_1 = segment_free_flow_speed(1201419, 1201469, median_speed_by_station)
v_ff_2 = segment_free_flow_speed(1201469, 1201497, median_speed_by_station)
v_ff_3 = segment_free_flow_speed(1201497, 1201525, median_speed_by_station)
v_ff_4 = segment_free_flow_speed(1201525, 1201558, median_speed_by_station)
v_ff_5 = segment_free_flow_speed(1201558, 1201589, median_speed_by_station)
v_ff_6 = segment_free_flow_speed(1201589, 1201620, median_speed_by_station)

segment_free_flow_df = pd.DataFrame({
    "segment": ["S1", "S2", "S3", "S4", "S5", "S6"],
    "from_station": ["RED HILL", "BRISTOL 1", "FAIRVIEW", "HARBOR 1", "HARBOR 2", "EUCLID"],
    "to_station": ["BRISTOL 1", "FAIRVIEW", "HARBOR 1", "HARBOR 2", "EUCLID", "TALBERT"],
    "v_ff_mph": [v_ff_1, v_ff_2, v_ff_3, v_ff_4, v_ff_5, v_ff_6]
})
segment_free_flow_df["v_ff_mph"] = segment_free_flow_df["v_ff_mph"].round(3)
display(segment_free_flow_df)

,segment,from_station,to_station,v_ff_mph
0,S1,RED HILL,BRISTOL 1,64.770
1,S2,BRISTOL 1,FAIRVIEW,67.525
2,S3,FAIRVIEW,HARBOR 1,69.117
3,S4,HARBOR 1,HARBOR 2,68.550
4,S5,HARBOR 2,EUCLID,68.373
5,S6,EUCLID,TALBERT,68.000


In [56]:
# 7. Free-flow travel time calculation
# Purpose:
# Convert each segment's free-flow speed into free-flow travel time.

# Formula:
# travel_time_hours = segment_length_miles / free_flow_speed_mph
# travel_time_minutes = travel_time_hours * 60
# segment length in miles

L_1 = 1.14
L_2 = 0.74
L_3 = 0.92
L_4 = 0.30
L_5 = 1.00
L_6 = 0.80

def segment_free_flow_travel_time(length, segment_free_flow_speed):
    travel_time_hour = length / segment_free_flow_speed
    travel_time_min = travel_time_hour * 60
    return travel_time_hour, travel_time_min

TT_ff_1_hr, TT_ff_1_min = segment_free_flow_travel_time(L_1, v_ff_1)
TT_ff_2_hr, TT_ff_2_min = segment_free_flow_travel_time(L_2, v_ff_2)
TT_ff_3_hr, TT_ff_3_min = segment_free_flow_travel_time(L_3, v_ff_3)
TT_ff_4_hr, TT_ff_4_min = segment_free_flow_travel_time(L_4, v_ff_4)
TT_ff_5_hr, TT_ff_5_min = segment_free_flow_travel_time(L_5, v_ff_5)
TT_ff_6_hr, TT_ff_6_min = segment_free_flow_travel_time(L_6, v_ff_6)

total_h = TT_ff_1_hr + TT_ff_2_hr + TT_ff_3_hr + TT_ff_4_hr + TT_ff_5_hr + TT_ff_6_hr
total_min = TT_ff_1_min + TT_ff_2_min + TT_ff_3_min + TT_ff_4_min + TT_ff_5_min + TT_ff_6_min

free_flow_tt_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "length_miles": [L_1, L_2, L_3, L_4, L_5, L_6],
    "free_flow_speed_mph": [v_ff_1, v_ff_2, v_ff_3, v_ff_4, v_ff_5, v_ff_6],
    "TT_ff_hr": [TT_ff_1_hr, TT_ff_2_hr, TT_ff_3_hr, TT_ff_4_hr, TT_ff_5_hr, TT_ff_6_hr],
    "TT_ff_min": [TT_ff_1_min, TT_ff_2_min, TT_ff_3_min, TT_ff_4_min, TT_ff_5_min, TT_ff_6_min],
})

print("Segment Free-Flow Travel Time ")
display(free_flow_tt_df)

print("Total free-flow travel time:")
print("Hours:", round(total_h, 3))
print("Minutes:", round(total_min, 3))

Segment Free-Flow Travel Time 


,segment,length_miles,free_flow_speed_mph,TT_ff_hr,TT_ff_min
0,Segment 1: RED HILL → BRISTOL 1,1.14,64.769896,0.017601,1.056046
1,Segment 2: BRISTOL 1 → FAIRVIEW,0.74,67.525111,0.010959,0.657533
2,Segment 3: FAIRVIEW → HARBOR 1,0.92,69.117396,0.013311,0.798641
3,Segment 4: HARBOR 1 → HARBOR 2,0.30,68.549672,0.004376,0.262583
4,Segment 5: HARBOR 2 → EUCLID,1.00,68.373455,0.014626,0.877534
5,Segment 6: EUCLID → TALBERT,0.80,67.999963,0.011765,0.705883


Total free-flow travel time:
Hours: 0.073
Minutes: 4.358


In [57]:
# ============================================================
# Flow/speed-based sanity check: observed travel time at 08:00
# Purpose: This creates TT_obs_1_hr ... TT_obs_6_min needed for the flow/speed-based segment delay sanity check.

mainline_ids = [
    1201419,
    1201469,
    1201497,
    1201525,
    1201558,
    1201589,
    1201620
]


# Get mainline station speeds at exactly 08:00
mainline_ids_8am = selected_data_morning[
    (selected_data_morning["timestamp"] == pd.Timestamp("2026-01-08 08:00:00")) &
    (selected_data_morning["station_id"].isin(mainline_ids)) &
    (selected_data_morning["station_type"] == "ML")
].copy()


mainline_ids_8am["avg_speed"] = pd.to_numeric(
    mainline_ids_8am["avg_speed"],
    errors="coerce"
)


def get_station_speed(station_id, df):
    station_speed = df.loc[
        df["station_id"] == station_id,
        "avg_speed"
    ]

    if station_speed.empty:
        raise ValueError(f"Missing speed for station {station_id}")

    return station_speed.iloc[0]


def observed_segment_speed(upstream_id, downstream_id, df):
    v_upstream = get_station_speed(upstream_id, df)
    v_downstream = get_station_speed(downstream_id, df)

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)


# Observed segment speeds at 08:00
v_obs_1 = observed_segment_speed(1201419, 1201469, mainline_ids_8am)
v_obs_2 = observed_segment_speed(1201469, 1201497, mainline_ids_8am)
v_obs_3 = observed_segment_speed(1201497, 1201525, mainline_ids_8am)
v_obs_4 = observed_segment_speed(1201525, 1201558, mainline_ids_8am)
v_obs_5 = observed_segment_speed(1201558, 1201589, mainline_ids_8am)
v_obs_6 = observed_segment_speed(1201589, 1201620, mainline_ids_8am)


def segment_observed_travel_time(length, observed_segment_speed):
    travel_time_hour = length / observed_segment_speed
    travel_time_min = travel_time_hour * 60

    return travel_time_hour, travel_time_min


# Observed travel time by segment
TT_obs_1_hr, TT_obs_1_min = segment_observed_travel_time(L_1, v_obs_1)
TT_obs_2_hr, TT_obs_2_min = segment_observed_travel_time(L_2, v_obs_2)
TT_obs_3_hr, TT_obs_3_min = segment_observed_travel_time(L_3, v_obs_3)
TT_obs_4_hr, TT_obs_4_min = segment_observed_travel_time(L_4, v_obs_4)
TT_obs_5_hr, TT_obs_5_min = segment_observed_travel_time(L_5, v_obs_5)
TT_obs_6_hr, TT_obs_6_min = segment_observed_travel_time(L_6, v_obs_6)


observed_travel_time_sanity_check_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "observed_speed_mph": [
        v_obs_1,
        v_obs_2,
        v_obs_3,
        v_obs_4,
        v_obs_5,
        v_obs_6,
    ],
    "observed_travel_time_min": [
        TT_obs_1_min,
        TT_obs_2_min,
        TT_obs_3_min,
        TT_obs_4_min,
        TT_obs_5_min,
        TT_obs_6_min,
    ],
    "observed_travel_time_hr": [
        TT_obs_1_hr,
        TT_obs_2_hr,
        TT_obs_3_hr,
        TT_obs_4_hr,
        TT_obs_5_hr,
        TT_obs_6_hr,
    ],
})

print("=== Flow/Speed-Based Observed Travel Time Sanity Check ===")
display(observed_travel_time_sanity_check_df.round(3))

=== Flow/Speed-Based Observed Travel Time Sanity Check ===


,segment,observed_speed_mph,observed_travel_time_min,observed_travel_time_hr
0,Segment 1: RED HILL → BRISTOL 1,39.698,1.723,0.029
1,Segment 2: BRISTOL 1 → FAIRVIEW,50.776,0.874,0.015
2,Segment 3: FAIRVIEW → HARBOR 1,42.492,1.299,0.022
3,Segment 4: HARBOR 1 → HARBOR 2,29.078,0.619,0.010
4,Segment 5: HARBOR 2 → EUCLID,24.708,2.428,0.040
5,Segment 6: EUCLID → TALBERT,26.840,1.788,0.030


# flow-based travel-time benchmark
This calculation is included only as a PeMS speed-based sanity check.
It estimates per-vehicle corridor delay using observed PeMS speeds and free-flow travel time.
It is not used as the final benchmark objective because it does not include CTM states, ramp queues, fairness penalties, or capacity/spillback penalties.

In [58]:

# Flow/speed-based sanity check: segment delay
# Purpose:
# This calculates per-vehicle delay using:
#     delay = observed travel time - free-flow travel time

def segment_delay(segment_observed_travel_time, segment_free_flow_travel_time):
    segment_delay = segment_observed_travel_time - segment_free_flow_travel_time
    return segment_delay


# Segment delay in hours
delay_1 = segment_delay(TT_obs_1_hr, TT_ff_1_hr)
delay_2 = segment_delay(TT_obs_2_hr, TT_ff_2_hr)
delay_3 = segment_delay(TT_obs_3_hr, TT_ff_3_hr)
delay_4 = segment_delay(TT_obs_4_hr, TT_ff_4_hr)
delay_5 = segment_delay(TT_obs_5_hr, TT_ff_5_hr)
delay_6 = segment_delay(TT_obs_6_hr, TT_ff_6_hr)



# Segment delay in minutes
delay_1_min = segment_delay(TT_obs_1_min, TT_ff_1_min)
delay_2_min = segment_delay(TT_obs_2_min, TT_ff_2_min)
delay_3_min = segment_delay(TT_obs_3_min, TT_ff_3_min)
delay_4_min = segment_delay(TT_obs_4_min, TT_ff_4_min)
delay_5_min = segment_delay(TT_obs_5_min, TT_ff_5_min)
delay_6_min = segment_delay(TT_obs_6_min, TT_ff_6_min)


# Total corridor delay

total_delay_hours = (
    delay_1
    + delay_2
    + delay_3
    + delay_4
    + delay_5
    + delay_6
)

total_delay_min = (
    delay_1_min
    + delay_2_min
    + delay_3_min
    + delay_4_min
    + delay_5_min
    + delay_6_min
)


# Clean display table
segment_delay_sanity_check_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "observed_travel_time_min": [
        TT_obs_1_min,
        TT_obs_2_min,
        TT_obs_3_min,
        TT_obs_4_min,
        TT_obs_5_min,
        TT_obs_6_min,
    ],
    "free_flow_travel_time_min": [
        TT_ff_1_min,
        TT_ff_2_min,
        TT_ff_3_min,
        TT_ff_4_min,
        TT_ff_5_min,
        TT_ff_6_min,
    ],
    "delay_min": [
        delay_1_min,
        delay_2_min,
        delay_3_min,
        delay_4_min,
        delay_5_min,
        delay_6_min,
    ],
    "delay_hr": [
        delay_1,
        delay_2,
        delay_3,
        delay_4,
        delay_5,
        delay_6,
    ],
})

print(" Flow/Speed-Based Segment Delay Sanity Check ")
display(segment_delay_sanity_check_df.round(3))

print("Total delay hours:", round(total_delay_hours, 3))
print("Total delay minutes:", round(total_delay_min, 3))

 Flow/Speed-Based Segment Delay Sanity Check 


,segment,observed_travel_time_min,free_flow_travel_time_min,delay_min,delay_hr
0,Segment 1: RED HILL → BRISTOL 1,1.723,1.056,0.667,0.011
1,Segment 2: BRISTOL 1 → FAIRVIEW,0.874,0.658,0.217,0.004
2,Segment 3: FAIRVIEW → HARBOR 1,1.299,0.799,0.500,0.008
3,Segment 4: HARBOR 1 → HARBOR 2,0.619,0.263,0.356,0.006
4,Segment 5: HARBOR 2 → EUCLID,2.428,0.878,1.551,0.026
5,Segment 6: EUCLID → TALBERT,1.788,0.706,1.083,0.018


Total delay hours: 0.073
Total delay minutes: 4.374


In [59]:
# Total mainline delay for each segment
# delay_i is in vehicle-hours per vehicle
# flow_i is vehicles during the 5-minute interval
# result is vehicle-hours during the 5-minute interval
def total_mainline_delay(segment_delay, total_flow):
    total_mainline_delay = segment_delay * total_flow
    return total_mainline_delay


In [60]:
# total mainline delay
# Formula:  total_delay_i = delay_i * Q_out_i

def total_mainline_delay(segment_delay, total_flow):
    total_mainline_delay = segment_delay * total_flow
    return total_mainline_delay



# Q_out_i = downstream mainline station flow for segment i at 08:00
mainline_flow_8am = selected_data_morning[
    (selected_data_morning["timestamp"] == pd.Timestamp("2026-01-08 08:00:00")) &
    (selected_data_morning["station_type"] == "ML")
].copy()

mainline_flow_8am["total_flow"] = pd.to_numeric(
    mainline_flow_8am["total_flow"],
    errors="coerce"
)


def get_station_flow(station_id, df):
    station_flow = df.loc[
        df["station_id"] == station_id,
        "total_flow"
    ]

    if station_flow.empty:
        raise ValueError(f"Missing flow for station {station_id}")

    return station_flow.iloc[0]


# Downstream station flow for each segment
Q_out_1 = get_station_flow(1201469, mainline_flow_8am)  # BRISTOL 1
Q_out_2 = get_station_flow(1201497, mainline_flow_8am)  # FAIRVIEW
Q_out_3 = get_station_flow(1201525, mainline_flow_8am)  # HARBOR 1
Q_out_4 = get_station_flow(1201558, mainline_flow_8am)  # HARBOR 2
Q_out_5 = get_station_flow(1201589, mainline_flow_8am)  # EUCLID
Q_out_6 = get_station_flow(1201620, mainline_flow_8am)  # TALBERT


# Mainline delay in vehicle-minutes
mainline_delay_1_min = total_mainline_delay(delay_1_min, Q_out_1)
mainline_delay_2_min = total_mainline_delay(delay_2_min, Q_out_2)
mainline_delay_3_min = total_mainline_delay(delay_3_min, Q_out_3)
mainline_delay_4_min = total_mainline_delay(delay_4_min, Q_out_4)
mainline_delay_5_min = total_mainline_delay(delay_5_min, Q_out_5)
mainline_delay_6_min = total_mainline_delay(delay_6_min, Q_out_6)


# Mainline delay in vehicle-hours
mainline_delay_1_hr = mainline_delay_1_min / 60
mainline_delay_2_hr = mainline_delay_2_min / 60
mainline_delay_3_hr = mainline_delay_3_min / 60
mainline_delay_4_hr = mainline_delay_4_min / 60
mainline_delay_5_hr = mainline_delay_5_min / 60
mainline_delay_6_hr = mainline_delay_6_min / 60


# Total flow-based mainline delay
total_flow_based_delay_min = sum([
    mainline_delay_1_min,
    mainline_delay_2_min,
    mainline_delay_3_min,
    mainline_delay_4_min,
    mainline_delay_5_min,
    mainline_delay_6_min,
])

total_flow_based_delay_hr = total_flow_based_delay_min / 60


# Clean display table
flow_based_mainline_delay_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "Q_out_vehicles_5min": [
        Q_out_1,
        Q_out_2,
        Q_out_3,
        Q_out_4,
        Q_out_5,
        Q_out_6,
    ],
    "delay_min_per_vehicle": [
        delay_1_min,
        delay_2_min,
        delay_3_min,
        delay_4_min,
        delay_5_min,
        delay_6_min,
    ],
    "mainline_delay_vehicle_min": [
        mainline_delay_1_min,
        mainline_delay_2_min,
        mainline_delay_3_min,
        mainline_delay_4_min,
        mainline_delay_5_min,
        mainline_delay_6_min,
    ],
    "mainline_delay_vehicle_hr": [
        mainline_delay_1_hr,
        mainline_delay_2_hr,
        mainline_delay_3_hr,
        mainline_delay_4_hr,
        mainline_delay_5_hr,
        mainline_delay_6_hr,
    ],
})

print("=== Flow/Speed-Based Mainline Delay Sanity Check ===")
display(flow_based_mainline_delay_df.round(3))

print("Total flow-based delay in vehicle-min:", round(total_flow_based_delay_min, 3))
print("Total flow-based delay in vehicle-hr:", round(total_flow_based_delay_hr, 3))

=== Flow/Speed-Based Mainline Delay Sanity Check ===


,segment,Q_out_vehicles_5min,delay_min_per_vehicle,mainline_delay_vehicle_min,mainline_delay_vehicle_hr
0,Segment 1: RED HILL → BRISTOL 1,681.0,0.667,454.209,7.570
1,Segment 2: BRISTOL 1 → FAIRVIEW,622.0,0.217,134.906,2.248
2,Segment 3: FAIRVIEW → HARBOR 1,737.0,0.500,368.826,6.147
3,Segment 4: HARBOR 1 → HARBOR 2,599.0,0.356,213.510,3.559
4,Segment 5: HARBOR 2 → EUCLID,680.0,1.551,1054.537,17.576
5,Segment 6: EUCLID → TALBERT,550.0,1.083,595.384,9.923


Total flow-based delay in vehicle-min: 2821.373
Total flow-based delay in vehicle-hr: 47.023


# State based total delay calculation

---------------------------------------------------------------
6-cell, 5-min prototype state-based delay (Section 8.2 of note)
NOTE: superseded by the 8-cell, 30-sec model in Part 3.
X_final here uses observed Q_out (no CTM sending/receiving),
which is why this number CANNOT be compared to the
26532 vs 15047 veh-min totals from the 8-cell ADMM model.
Kept only as a sanity check of the state-based formula.


# This computes the state-based baseline that matches note:
$$D_M,i(t) = TTT_i(t) - (Q_out,i + f_i) * TT_ff,i$$

where:
$$TTT_i(t) = ((X_i,t-1 + X_i,t) / 2) * DeltaT$$

# Units:
X = vehicles

DeltaT = minutes

TTT = vehicle-minutes

TT_ff = minutes

$(Q_out + f_i) * TT_ff$ = vehicle-minutes

$D_M,i = vehicle-minutes$



In [61]:
#1. build 08:00–09:00 benchmark window
start_time = "2026-01-08 08:00:00"
end_time   = "2026-01-08 09:00:00"

filtered_data_morning = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

selected_data_morning = filtered_data_morning[[0, 1, 5, 9, 10, 11]].copy()

selected_data_morning.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

selected_data_morning["total_flow"] = pd.to_numeric(
    selected_data_morning["total_flow"],
    errors="coerce"
)

selected_data_morning = selected_data_morning.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print("Morning Benchmark Window")
print("Rows:", len(selected_data_morning))
print("Unique timestamps:", selected_data_morning["timestamp"].nunique())
print("Unique stations:", selected_data_morning["station_id"].nunique())

display(
    selected_data_morning
    .groupby("station_id")
    .size()
    .reset_index(name="row_count")
)

Morning Benchmark Window
Rows: 180
Unique timestamps: 12
Unique stations: 15


,station_id,row_count
0,1201419,12
1,1201460,12
2,1201465,12
3,1201469,12
4,1201490,12
5,1201497,12
6,1201517,12
7,1201525,12
8,1201548,12
9,1201554,12


# Build 120-step CTM input series from 08:00–09:00 PeMS data
Purpose:
Convert 5-minute PeMS flows into 30-second CTM inputs.

PeMS gives:
     1 value every 5 minutes

CTM needs:
     1 value every 30 seconds

Therefore:
     each 5-minute flow is divided by 10
     and repeated for ten 30-second CTM steps

This creates:
     q_in_boundary_series
     observed_release_series
     ramp_arrival_series
     f_out_series
These are used for the official 8-cell / 30-sec / 120-step state-based benchmark.


In [62]:
# 2. Build 120-step CTM input series from 08:00–09:00 PeMS data
num_steps = 120
delta_t = 0.5  # 30 seconds = 0.5 minutes


# Get one PeMS station's 5-minute flow  and turn it into a 120-step 30-second CTM.
def build_30sec_series_from_5min_flow(station_id, selected_data_morning):
    station_5min_data = selected_data_morning[
        selected_data_morning["station_id"] == station_id
    ].copy()

    station_5min_data = station_5min_data.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    station_5min_data["total_flow"] = pd.to_numeric(
        station_5min_data["total_flow"],
        errors="coerce"
    ).fillna(0.0)

    flow_30sec_series = []

    for flow_5min in station_5min_data["total_flow"]:
        flow_30sec = flow_5min / 10.0

        for _ in range(10):
            flow_30sec_series.append(flow_30sec)

    if len(flow_30sec_series) != num_steps:
        raise ValueError(
            f"Station {station_id} produced {len(flow_30sec_series)} steps, "
            f"but expected {num_steps}."
        )

    return flow_30sec_series


# Boundary inflow into Cell 1 , Station 1201419
q_in_boundary_series = build_30sec_series_from_5min_flow(
    1201419,
    selected_data_morning
)


# Observed on-ramp releases
# These are the no-control benchmark ramp releases.
observed_release_series = {
    "u1": build_30sec_series_from_5min_flow(1201460, selected_data_morning),  # BRISTOL 1 OR
    "u2": build_30sec_series_from_5min_flow(1201490, selected_data_morning),  # FAIRVIEW OR
    "u3": build_30sec_series_from_5min_flow(1201517, selected_data_morning),  # HARBOR 1 OR
    "u4": build_30sec_series_from_5min_flow(1201548, selected_data_morning),  # HARBOR 2 OR
    "u5": build_30sec_series_from_5min_flow(1201580, selected_data_morning),  # EUCLID OR
}


# Ramp arrivals
# arrival demand is arrival_multiplier times observed release.
# we made same assumption across all the calculations

arrival_multiplier = 1.5
ramp_arrival_series = {}
for ramp in observed_release_series:
    ramp_arrival_series[ramp] = [
        arrival_multiplier * value
        for value in observed_release_series[ramp]
    ]


# Off-ramp flows mapped into CTM cells
# Missing BRISTOL off-ramp values are filled as 0.

f_out_bristol_series = build_30sec_series_from_5min_flow(
    1201465,
    selected_data_morning
)

f_out_harbor2_series = build_30sec_series_from_5min_flow(
    1201554,
    selected_data_morning
)

f_out_euclid_series = build_30sec_series_from_5min_flow(
    1201585,
    selected_data_morning
)

f_out_series = []

for step in range(num_steps):
    f_out_series.append({
        "Cell 1": 0.0,
        "Cell 2": f_out_bristol_series[step],
        "Cell 3": 0.0,
        "Cell 4": 0.0,
        "Cell 5": 0.0,
        "Cell 6": f_out_harbor2_series[step],
        "Cell 7": f_out_euclid_series[step],
        "Cell 8": 0.0,
    })



In [63]:
# Clean input-series check
input_series_length_check_df = pd.DataFrame({
    "series_name": [
        "q_in_boundary_series",
        "f_out_series",
        "u1 observed_release",
        "u2 observed_release",
        "u3 observed_release",
        "u4 observed_release",
        "u5 observed_release",
    ],
    "length": [
        len(q_in_boundary_series),
        len(f_out_series),
        len(observed_release_series["u1"]),
        len(observed_release_series["u2"]),
        len(observed_release_series["u3"]),
        len(observed_release_series["u4"]),
        len(observed_release_series["u5"]),
    ],
})

first_step_input_df = pd.DataFrame({
    "input_type": [
        "boundary inflow",
        "observed release",
        "observed release",
        "observed release",
        "observed release",
        "observed release",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
    ],
    "location": [
        "Cell 1 boundary",
        "u1",
        "u2",
        "u3",
        "u4",
        "u5",
        "u1",
        "u2",
        "u3",
        "u4",
        "u5",
        "Cell 1",
        "Cell 2",
        "Cell 3",
        "Cell 4",
        "Cell 5",
        "Cell 6",
        "Cell 7",
        "Cell 8",
    ],
    "first_30sec_value": [
        q_in_boundary_series[0],
        observed_release_series["u1"][0],
        observed_release_series["u2"][0],
        observed_release_series["u3"][0],
        observed_release_series["u4"][0],
        observed_release_series["u5"][0],
        ramp_arrival_series["u1"][0],
        ramp_arrival_series["u2"][0],
        ramp_arrival_series["u3"][0],
        ramp_arrival_series["u4"][0],
        ramp_arrival_series["u5"][0],
        f_out_series[0]["Cell 1"],
        f_out_series[0]["Cell 2"],
        f_out_series[0]["Cell 3"],
        f_out_series[0]["Cell 4"],
        f_out_series[0]["Cell 5"],
        f_out_series[0]["Cell 6"],
        f_out_series[0]["Cell 7"],
        f_out_series[0]["Cell 8"],
    ],
})

print("120-Step CTM Input Series Length Check")
display(input_series_length_check_df)
print(" First 30-Second CTM Input Values ")
display(first_step_input_df.round(3))

120-Step CTM Input Series Length Check


,series_name,length
0,q_in_boundary_series,120
1,f_out_series,120
2,u1 observed_release,120
3,u2 observed_release,120
4,u3 observed_release,120
5,u4 observed_release,120
6,u5 observed_release,120


 First 30-Second CTM Input Values 


,input_type,location,first_30sec_value
0,boundary inflow,Cell 1 boundary,64.50
1,observed release,u1,8.80
2,observed release,u2,3.90
3,observed release,u3,7.30
4,observed release,u4,5.60
5,observed release,u5,6.40
6,ramp arrival,u1,13.20
7,ramp arrival,u2,5.85
8,ramp arrival,u3,10.95
9,ramp arrival,u4,8.40


## Ramp Maximum Queue Capacity

This block estimates the maximum number of vehicles that each on-ramp can physically store before spillback occurs.

The maximum ramp queue is calculated using:

$$
R_{max,i} = \frac{L_i \times n_i}{l_{veh}}
$$

where:

- \(R_{max,i}\) = maximum queue storage for ramp \(i\), measured in vehicles
- \(L_i\) = ramp length in feet
- \(n_i\) = number of ramp lanes
- \(l_{veh}\) = assumed average vehicle length in feet

In this model, the average vehicle length is assumed to be:

$$
l_{veh} = 25 \text{ ft/vehicle}
$$

The calculated ramp storage capacities are stored in three formats:

1. `ramp_max_queue_named`
   Uses real ramp names such as `"Bristol 1"` and `"Fairview"`.
   This is used by the fairness penalty calculation.

2. `ramp_max_queue_by_u`
   Uses control variable names such as `"u1"` to `"u5"`.
   This is used by the ramp queue and spillback calculations.

3. `ramp_name_map`
   Maps each control variable name to its real ramp name.

These ramp capacity values are later used in the state-based CTM benchmark to calculate ramp stress, fairness penalty, and spillback.

In [64]:
# 3. Ramp maximum queue capacity for fairness penalty
# Purpose: Estimate maximum ramp queue storage using:
ave_veh_length = 25  # feet per vehicle


ramp_length = {
    "Bristol 1": 716.73,
    "Fairview": 1808.89,
    "Harbor 1": 1404.20,
    "Harbor 2": 1811.02,
    "Euclid": 610.24,
}


number_of_lanes = {
    "Bristol 1": 1,
    "Fairview": 1,
    "Harbor 1": 1,
    "Harbor 2": 1,
    "Euclid": 2,
}


def R_max(ave_veh_length, ramp_length, number_of_lanes):
    R_max = {}

    for ramps in ramp_length:
        R_max[ramps] = (
            ramp_length[ramps] * number_of_lanes[ramps]
        ) / ave_veh_length

    return R_max


R_max_value = R_max(
    ave_veh_length,
    ramp_length,
    number_of_lanes
)


# Named ramp max queue Used by fairness penalty
ramp_max_queue_named = {
    "Bristol 1": R_max_value["Bristol 1"],
    "Fairview": R_max_value["Fairview"],
    "Harbor 1": R_max_value["Harbor 1"],
    "Harbor 2": R_max_value["Harbor 2"],
    "Euclid": R_max_value["Euclid"],
}



# Ramp max queue using u1–u5 names
# Used by ramp queue / spillback calculation
ramp_max_queue_by_u = {
    "u1": R_max_value["Bristol 1"],
    "u2": R_max_value["Fairview"],
    "u3": R_max_value["Harbor 1"],
    "u4": R_max_value["Harbor 2"],
    "u5": R_max_value["Euclid"],
}


# Map u1–u5 control names to real ramp names
# Used by fairness_penalty_one_step(...)
ramp_name_map = {
    "u1": "Bristol 1",
    "u2": "Fairview",
    "u3": "Harbor 1",
    "u4": "Harbor 2",
    "u5": "Euclid",
}

# Clean display

ramp_capacity_df = pd.DataFrame({
    "ramp": list(ramp_max_queue_named.keys()),
    "ramp_length_ft": [
        ramp_length["Bristol 1"],
        ramp_length["Fairview"],
        ramp_length["Harbor 1"],
        ramp_length["Harbor 2"],
        ramp_length["Euclid"],
    ],
    "lanes": [
        number_of_lanes["Bristol 1"],
        number_of_lanes["Fairview"],
        number_of_lanes["Harbor 1"],
        number_of_lanes["Harbor 2"],
        number_of_lanes["Euclid"],
    ],
    "max_queue_vehicles": [
        ramp_max_queue_named["Bristol 1"],
        ramp_max_queue_named["Fairview"],
        ramp_max_queue_named["Harbor 1"],
        ramp_max_queue_named["Harbor 2"],
        ramp_max_queue_named["Euclid"],
    ],
})

print("=== Ramp Maximum Queue Capacity ===")
display(ramp_capacity_df.round(3))

=== Ramp Maximum Queue Capacity ===


,ramp,ramp_length_ft,lanes,max_queue_vehicles
0,Bristol 1,716.73,1,28.669
1,Fairview,1808.89,1,72.356
2,Harbor 1,1404.20,1,56.168
3,Harbor 2,1811.02,1,72.441
4,Euclid,610.24,2,48.819


## Fairness Penalty and Ramp Queue Spillback Functions

This section defines the fairness penalty and ramp queue update functions used in the official state-based CTM benchmark.

The benchmark uses the same fairness penalty structure as the ADMM and ADMM-MPC models.

### Fairness Penalty

For each ramp, the queue stress index is calculated as:

$$
\phi_i(t) = \min\left(\frac{R_i(t)}{R_{max,i}}, 1\right)
$$

where:

- \(R_i(t)\) = ramp queue at time step \(t\)
- \(R_{max,i}\) = maximum storage capacity of ramp \(i\)
- \(\phi_i(t)\) = capped ramp stress index

The stress value is capped at 1 so that any queue above physical storage capacity is treated as full saturation.

The fairness penalty is:

$$
L_{fair}(t) = \gamma \sum_{i<j} \left(\phi_i(t) - \phi_j(t)\right)^2
$$

where:

- \(\gamma\) = fairness penalty weight
- \(\phi_i(t)\) and \(\phi_j(t)\) = capped stress values for two different ramps

This penalty increases when one ramp is much more congested than another ramp.

---

### Ramp Queue Update with Spillback

Each ramp queue is updated using:

$$
R_i(t+1) = R_i(t) + a_i(t) - u_i(t)
$$

where:

- \(R_i(t)\) = current ramp queue
- \(a_i(t)\) = ramp arrival demand
- \(u_i(t)\) = observed ramp release
- \(R_i(t+1)\) = next ramp queue

The uncapped queue is first calculated as:

$$
R_{uncapped,i}(t+1) = R_i(t) + a_i(t) - u_i(t)
$$

The queue cannot be negative:

$$
R_{uncapped,i}(t+1) = \max(R_{uncapped,i}(t+1), 0)
$$

If the uncapped queue exceeds ramp storage capacity, the excess vehicles are counted as spillback:

$$
spillback_i(t) = \max(R_{uncapped,i}(t+1) - R_{max,i}, 0)
$$

The stored ramp queue is then capped at the maximum ramp capacity:

$$
R_i(t+1) = \min(R_{uncapped,i}(t+1), R_{max,i})
$$

These functions are used at every 30-second CTM step in the state-based benchmark simulation.

In [65]:
# 4. Fairness penalty setup for CTM benchmark
#fairness multiplayer (same value used for all the set ups)
gamma = 1

def fairness_penalty_one_step(
    R_next,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
):

    # 1. Compute capped stress index for each ramp
    stress_dict = {}

    for u_name, R_t in R_next.items():
        ramp_name = ramp_name_map[u_name]
        R_max_i = ramp_max_queue_named[ramp_name]

        raw_stress = R_t / R_max_i
        capped_stress = min(raw_stress, 1.0)

        stress_dict[ramp_name] = capped_stress


    # 2. Compute pairwise fairness penalty
    ramps = list(stress_dict.keys())
    fairness_sum = 0.0

    for i in range(len(ramps)):
        for j in range(i + 1, len(ramps)):
            phi_i = stress_dict[ramps[i]]
            phi_j = stress_dict[ramps[j]]

            fairness_sum += (phi_i - phi_j) ** 2


    # 3. Apply fairness weight
    L_fair = gamma * fairness_sum
    return stress_dict, fairness_sum, L_fair

In [66]:
# 5. Ramp queue update with spillback
# Purpose: Update each ramp queue using:

def ramp_next_queue_with_spillback(
    R_current,
    ramp_arrival_step,
    observed_release_step,
    ramp_max_queue_by_u
):
    R_next = {}
    spillback_by_ramp = {}

    for ramp in R_current:

        # Raw next queue before capacity cap
        R_uncapped = (
            R_current[ramp]
            + ramp_arrival_step[ramp]
            - observed_release_step[ramp]
        )

        # Queue cannot be negative
        R_uncapped = max(R_uncapped, 0.0)

        # Spillback happens if queue exceeds ramp max storage
        spillback = max(
            R_uncapped - ramp_max_queue_by_u[ramp],
            0.0
        )

        # Cap stored queue at physical ramp storage
        R_next[ramp] = min(
            R_uncapped,
            ramp_max_queue_by_u[ramp]
        )

        spillback_by_ramp[ramp] = spillback

    return R_next, spillback_by_ramp

In [67]:
# 6. Fairness penalty setup check
# Purpose: Test fairness_penalty_one_step using the first 30-second CTM step.

# Use first 30-second benchmark step
fairness_test_step = 0

# If ramp_queue_0 is not already defined, start queues at zero
if "ramp_queue_0" not in globals():
    ramp_queue_0 = {
        "u1": 0.0,
        "u2": 0.0,
        "u3": 0.0,
        "u4": 0.0,
        "u5": 0.0,
    }


R_current_test = ramp_queue_0.copy()

ramp_arrival_step_test = {
    "u1": ramp_arrival_series["u1"][fairness_test_step],
    "u2": ramp_arrival_series["u2"][fairness_test_step],
    "u3": ramp_arrival_series["u3"][fairness_test_step],
    "u4": ramp_arrival_series["u4"][fairness_test_step],
    "u5": ramp_arrival_series["u5"][fairness_test_step],
}


observed_release_step_test = {
    "u1": observed_release_series["u1"][fairness_test_step],
    "u2": observed_release_series["u2"][fairness_test_step],
    "u3": observed_release_series["u3"][fairness_test_step],
    "u4": observed_release_series["u4"][fairness_test_step],
    "u5": observed_release_series["u5"][fairness_test_step],
}


# next ramp queue for first step
R_next_test, spillback_by_ramp_test = ramp_next_queue_with_spillback(
    R_current_test,
    ramp_arrival_step_test,
    observed_release_step_test,
    ramp_max_queue_by_u
)


# fairness penalty for first step
stress_dict_test, fairness_sum_test, L_fair_test = fairness_penalty_one_step(
    R_next_test,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
)


# result
fairness_check_rows = []
for u_name in R_next_test:
    ramp_name = ramp_name_map[u_name]
    R_max_i = ramp_max_queue_named[ramp_name]

    raw_stress = R_next_test[u_name] / R_max_i
    capped_stress = min(raw_stress, 1.0)

    fairness_check_rows.append({
        "u_name": u_name,
        "ramp_name": ramp_name,
        "R_prev": R_current_test[u_name],
        "arrival": ramp_arrival_step_test[u_name],
        "observed_release": observed_release_step_test[u_name],
        "R_next": R_next_test[u_name],
        "R_max": R_max_i,
        "raw_stress": raw_stress,
        "capped_stress": capped_stress,
        "spillback": spillback_by_ramp_test[u_name],
    })


fairness_check_df = pd.DataFrame(fairness_check_rows)
print(" Fairness Penalty First-Step Check")
display(fairness_check_df.round(3))

print("gamma:", gamma)
print("fairness_sum:", round(fairness_sum_test, 3))
print("L_fair:", round(L_fair_test, 3))

 Fairness Penalty First-Step Check


,u_name,ramp_name,R_prev,arrival,observed_release,R_next,R_max,raw_stress,capped_stress,spillback
0,u1,Bristol 1,0.0,13.20,8.8,4.40,28.669,0.153,0.153,0.0
1,u2,Fairview,0.0,5.85,3.9,1.95,72.356,0.027,0.027,0.0
2,u3,Harbor 1,0.0,10.95,7.3,3.65,56.168,0.065,0.065,0.0
3,u4,Harbor 2,0.0,8.40,5.6,2.80,72.441,0.039,0.039,0.0
4,u5,Euclid,0.0,9.60,6.4,3.20,48.819,0.066,0.066,0.0


gamma: 1
fairness_sum: 0.049
L_fair: 0.049


### Doorway Capacity Calculation

The doorway capacity represents the maximum number of vehicles that can pass through each mainline cell during one CTM timestep. In the PeMS data, the available doorway capacities are given at selected detector stations in units of vehicles per 5-minute interval.

Because the final CTM model uses a 30-second timestep, these capacities must be converted from station-level 5-minute values into cell-level 30-second values.

First, the per-lane doorway capacity is calculated for stations where capacity is available:

$$
C_{\text{per-lane},s}
=
\frac{C_s}{N_s}
$$

where:

- $C_s$ = doorway capacity at station $s$ in vehicles per 5 minutes
- $N_s$ = number of mainline lanes at station $s$
- $C_{\text{per-lane},s}$ = per-lane capacity in vehicles per 5 minutes

Since RED HILL and FAIRVIEW do not have given doorway capacities, their capacities are estimated using the average per-lane doorway capacity from the stations with known values:

$$
\bar{C}_{\text{per-lane}}
=
\frac{1}{n}
\sum_{s=1}^{n}
C_{\text{per-lane},s}
$$

Then the missing station capacity is estimated as:

$$
C_s
=
\bar{C}_{\text{per-lane}}
\cdot N_s
$$

After estimating the missing station capacities, the average per-lane capacity is converted from vehicles per 5 minutes to vehicles per 30 seconds:

$$
C_{\text{per-lane},30s}
=
\frac{\bar{C}_{\text{per-lane}}}{5}
\cdot \Delta t
$$

where:

- $\Delta t = 0.5$ minutes
- $C_{\text{per-lane},30s}$ = per-lane capacity in vehicles per 30 seconds

Finally, the doorway capacity for each CTM cell is computed as:

$$
C_i
=
C_{\text{per-lane},30s}
\cdot N_i
$$

where:

- $C_i$ = doorway capacity of CTM Cell $i$ in vehicles per 30 seconds
- $N_i$ = number of lanes in CTM Cell $i$

These final cell-level capacities are used inside the 8-cell, 30-second CTM model as the doorway capacity constraint. The exact values are kept in the model, while rounded values are shown only for display.

In [68]:
#7 . Doorway capacity setup for 8-cell / 30-sec CTM
# Purpose:Estimate mainline doorway capacity and convert it into the

doorway_capacity_given = {
    "Bristol 1": 895,
    "Harbor 1": 1029,
    "Harbor 2": 860,
    "Euclid": 925,
    "Talbert": 833
}

# Mainline lane count at detector stations
number_of_lanes = {
    "RED HILL": 5,
    "Bristol 1": 5,
    "FAIRVIEW": 5,
    "Harbor 1": 6,
    "Harbor 2": 5,
    "Euclid": 5,
    "Talbert": 5
}


In [69]:

# Compute per-lane doorway capacity
def per_lane_doorway_cap(doorway_capacity_dict, lane_dict):
    per_lane = {}

    for station in doorway_capacity_dict:
        per_lane[station] = (
            doorway_capacity_dict[station] / lane_dict[station]
        )

    return per_lane


def average_per_lane_capacity(per_lane_dict):
    total = 0.0

    for station in per_lane_dict:
        total += per_lane_dict[station]

    return total / len(per_lane_dict)


def doorway_capacity_not_given(avg_per_lane_cap, lane_dict, missing_stations):
    estimated = {}

    for station in missing_stations:
        estimated[station] = avg_per_lane_cap * lane_dict[station]

    return estimated


per_lane_capacity = per_lane_doorway_cap(
    doorway_capacity_given,
    number_of_lanes
)

avg_per_lane_cap = average_per_lane_capacity(per_lane_capacity)

missing_stations = ["RED HILL", "FAIRVIEW"]

estimated_capacity = doorway_capacity_not_given(
    avg_per_lane_cap,
    number_of_lanes,
    missing_stations
)


In [70]:
# Combine known and estimated station capacities
doorway_capacity_station_5min = {}

for station in doorway_capacity_given:
    doorway_capacity_station_5min[station] = doorway_capacity_given[station]

for station in estimated_capacity:
    doorway_capacity_station_5min[station] = estimated_capacity[station]


In [71]:
# Convert average per-lane capacity to CTM 30-sec units
delta_t = 0.5  # 30 seconds = 0.5 minutes

doorway_capacity_per_lane_per_min = avg_per_lane_cap / 5.0
doorway_capacity_per_lane_30sec = doorway_capacity_per_lane_per_min * delta_t

# 8-cell lane count

cell_lane_count = {
    1: 5,
    2: 5,
    3: 5,
    4: 5,
    5: 6,
    6: 5,
    7: 5,
    8: 5,
}


In [72]:
# Final CTM doorway capacity
doorway_capacity = {
    f"Cell {i}": doorway_capacity_per_lane_30sec * cell_lane_count[i]
    for i in range(1, 9)
}

In [73]:
# results

doorway_station_capacity_df = pd.DataFrame({
    "station": list(doorway_capacity_station_5min.keys()),
    "lanes": [
        number_of_lanes[station]
        for station in doorway_capacity_station_5min
    ],
    "doorway_capacity_veh_5min": [
        doorway_capacity_station_5min[station]
        for station in doorway_capacity_station_5min
    ],
    "per_lane_capacity_veh_5min": [
        doorway_capacity_station_5min[station] / number_of_lanes[station]
        for station in doorway_capacity_station_5min
    ],
})

doorway_cell_capacity_df = pd.DataFrame({
    "cell": list(doorway_capacity.keys()),
    "lanes": [
        cell_lane_count[i]
        for i in range(1, 9)
    ],
    "doorway_capacity_veh_30sec": [
        doorway_capacity[f"Cell {i}"]
        for i in range(1, 9)
    ],
})

print("Doorway Capacity from PeMS Stations ")
display(doorway_station_capacity_df.round(3))

print("Average per-lane doorway capacity, veh/5-min:", round(avg_per_lane_cap, 3))
print("Average per-lane doorway capacity, veh/min:", round(doorway_capacity_per_lane_per_min, 3))
print("Average per-lane doorway capacity, veh/30-sec:", round(doorway_capacity_per_lane_30sec, 3))

print("\n=== Final CTM Doorway Capacity by Cell ===")
display(doorway_cell_capacity_df.round(3))

Doorway Capacity from PeMS Stations 


,station,lanes,doorway_capacity_veh_5min,per_lane_capacity_veh_5min
0,Bristol 1,5,895.0,179.00
1,Harbor 1,6,1029.0,171.50
2,Harbor 2,5,860.0,172.00
3,Euclid,5,925.0,185.00
4,Talbert,5,833.0,166.60
5,RED HILL,5,874.1,174.82
6,FAIRVIEW,5,874.1,174.82


Average per-lane doorway capacity, veh/5-min: 174.82
Average per-lane doorway capacity, veh/min: 34.964
Average per-lane doorway capacity, veh/30-sec: 17.482

=== Final CTM Doorway Capacity by Cell ===


,cell,lanes,doorway_capacity_veh_30sec
0,Cell 1,5,87.410
1,Cell 2,5,87.410
2,Cell 3,5,87.410
3,Cell 4,5,87.410
4,Cell 5,6,104.892
5,Cell 6,5,87.410
6,Cell 7,5,87.410
7,Cell 8,5,87.410


### Physical Capacity Setup for the 8-Cell CTM

The old prototype calculated physical capacity for the original 6 detector-to-detector freeway segments. That version is no longer used for the official benchmark because the final CTM model uses 8 equal-length cells with a 30-second timestep.

For the official state-based CTM benchmark, physical capacity is calculated for each CTM cell using:

$$
N^{\max}_i = k_{\text{jam}} \cdot L_i \cdot n_i
$$

where:

- $N^{\max}_i$ = physical capacity of Cell $i$ in vehicles
- $k_{\text{jam}}$ = jam density, in vehicles per mile per lane
- $L_i$ = CTM cell length, in miles
- $n_i$ = number of lanes in Cell $i$

The total corridor length is:

$$
13.07 - 8.17 = 4.90 \text{ miles}
$$

Since the final CTM uses 8 equal cells:

$$
L_i = \frac{4.90}{8} = 0.6125 \text{ miles}
$$

Using a jam density of:

$$
k_{\text{jam}} = 193 \text{ veh/mi/lane}
$$

the physical capacity for a 5-lane cell is:

$$
193 \cdot 0.6125 \cdot 5 = 591.0625 \text{ vehicles}
$$

For Cell 5, which has 6 lanes:

$$
193 \cdot 0.6125 \cdot 6 = 709.275 \text{ vehicles}
$$

These physical capacity values are used later in the CTM state update and capacity penalty calculation.

In [74]:
# 8. Physical capacity setup for 8-cell CTM
# Purpose: Compute the maximum number of vehicles each CTM cell can store.

# from Highway Traffic Manual
jam_density = 193  # veh/mi/lane

# 8-cell corridor
segment_start_PM = 8.17
segment_end_PM = 13.07
num_cells = 8

cell_length = (segment_end_PM - segment_start_PM) / num_cells

# Number of lanes
number_of_lanes = {
    "Cell 1": 5,
    "Cell 2": 5,
    "Cell 3": 5,
    "Cell 4": 5,
    "Cell 5": 6,
    "Cell 6": 5,
    "Cell 7": 5,
    "Cell 8": 5,
}


In [75]:
# Physical capacity calculation
def calculate_physical_capacity(jam_density, cell_length, number_of_lanes):
    N_max_value = {}

    for cell in number_of_lanes:
        N_max_value[cell] = (
            jam_density
            * cell_length
            * number_of_lanes[cell]
        )

    return N_max_value


N_max_value = calculate_physical_capacity(
    jam_density,
    cell_length,
    number_of_lanes
)


# Final CTM physical capacity dictionary
physical_capacity = N_max_value.copy()


In [76]:
# Result
physical_capacity_df = pd.DataFrame({
    "cell": list(physical_capacity.keys()),
    "cell_length_miles": [cell_length for _ in physical_capacity],
    "lanes": [
        number_of_lanes[cell]
        for cell in physical_capacity
    ],
    "jam_density_veh_mi_lane": [
        jam_density
        for _ in physical_capacity
    ],
    "physical_capacity_vehicles": [
        physical_capacity[cell]
        for cell in physical_capacity
    ],
})

print("Physical Capacity by 8-Cell CTM Cell ")
print("cell_length:", round(cell_length, 4), "miles")
display(physical_capacity_df.round(3))

Physical Capacity by 8-Cell CTM Cell 
cell_length: 0.6125 miles


,cell,cell_length_miles,lanes,jam_density_veh_mi_lane,physical_capacity_vehicles
0,Cell 1,0.612,5,193,591.062
1,Cell 2,0.612,5,193,591.062
2,Cell 3,0.612,5,193,591.062
3,Cell 4,0.612,5,193,591.062
4,Cell 5,0.612,6,193,709.275
5,Cell 6,0.612,5,193,591.062
6,Cell 7,0.612,5,193,591.062
7,Cell 8,0.612,5,193,591.062


### Safe Occupancy Threshold for 8-Cell CTM

The safe occupancy threshold defines the maximum desired vehicle storage level for each CTM cell before the model begins applying a safety-related capacity penalty.

For each cell, the safe threshold is calculated as:

$$
X_{\text{safe}, i} = \eta \cdot N_{\max, i}
$$

where:

$$
X_{\text{safe}, i}
$$

is the safe occupancy threshold for Cell \(i\),

$$
\eta
$$

is the safe occupancy multiplier,

and

$$
N_{\max, i}
$$

is the physical capacity of Cell \(i\).

In this project, the official benchmark uses:

$$
\eta = 0.7
$$

because the ADMM and ADMM-MPC models also use the same threshold value.

This keeps the benchmark comparison consistent across all models:

- same 8-cell CTM structure
- same 30-second timestep
- same physical capacity values
- same safe occupancy threshold
- same capacity penalty definition

Therefore, the safe threshold used in the benchmark is:

$$
X_{\text{safe}, i} = 0.7 \cdot N_{\max, i}
$$

This value is later used inside the CTM objective through the safe threshold penalty term.

In [77]:
# 9. Safe occupancy threshold setup for 8-cell CTM
# Purpose:Define the safe occupancy threshold for each CTM cell.

#safe threshold multiple . same value used for all the set ups
eta = 0.7
safe_threshold_capacity = {}

for cell in N_max_value:
    safe_threshold_capacity[cell] = eta * N_max_value[cell]


safe_threshold_capacity_df = pd.DataFrame({
    "cell": list(safe_threshold_capacity.keys()),
    "eta": [eta for _ in safe_threshold_capacity],
    "physical_capacity": [
        N_max_value[cell]
        for cell in safe_threshold_capacity
    ],
    "safe_threshold_capacity": [
        safe_threshold_capacity[cell]
        for cell in safe_threshold_capacity
    ],
})

print(" Safe Threshold Capacity")
display(safe_threshold_capacity_df.round(3))

 Safe Threshold Capacity


,cell,eta,physical_capacity,safe_threshold_capacity
0,Cell 1,0.7,591.062,413.744
1,Cell 2,0.7,591.062,413.744
2,Cell 3,0.7,591.062,413.744
3,Cell 4,0.7,591.062,413.744
4,Cell 5,0.7,709.275,496.492
5,Cell 6,0.7,591.062,413.744
6,Cell 7,0.7,591.062,413.744
7,Cell 8,0.7,591.062,413.744


### Capacity Penalty for 8-Cell CTM

The capacity penalty is used to discourage physically unrealistic or unsafe traffic states in the CTM model.

The official benchmark uses the same capacity penalty structure as the ADMM and ADMM-MPC models.

The total capacity penalty has four components:

$$
L_{\text{cap}} =
L_{\text{doorway}}
+
L_{\text{safe}}
+
L_{\text{physical}}
+
L_{\text{spillback}}
$$

---

#### 1. Doorway Capacity Penalty

The doorway capacity penalty checks whether the total inflow entering a cell exceeds the cell doorway capacity.

For each cell:

$$
L_{\text{doorway}, i}
=
\lambda_1
\left[
\max(0, q_{\text{in}, i} + u_i - C_i)
\right]^2
$$

where:

$$
q_{\text{in}, i}
$$

is the mainline inflow into Cell \(i\),

$$
u_i
$$

is the on-ramp inflow into Cell \(i\),

and

$$
C_i
$$

is the doorway capacity of Cell \(i\).

---

#### 2. Safe Threshold Penalty

The safe threshold penalty checks whether the CTM state exceeds the desired safe occupancy level.

For each cell:

$$
L_{\text{safe}, i}
=
\lambda_2
\left[
\max(0, x_i - X_{\text{safe}, i})
\right]^2
$$

where:

$$
x_i
$$

is the vehicle state of Cell \(i\),

and

$$
X_{\text{safe}, i}
$$

is the safe occupancy threshold of Cell \(i\).

---

#### 3. Physical Capacity Penalty

The physical capacity penalty checks whether the CTM state exceeds the physical storage capacity of the cell.

For each cell:

$$
L_{\text{physical}, i}
=
\lambda_3
\left[
\max(0, x_i - N_{\max, i})
\right]^2
$$

where:

$$
N_{\max, i}
$$

is the physical vehicle capacity of Cell \(i\).

---

#### 4. Spillback Penalty

The spillback penalty checks whether a ramp queue exceeds the maximum ramp storage capacity.

For each ramp:

$$
L_{\text{spillback}, r}
=
\lambda_4
\left[
\text{spillback}_r
\right]^2
$$

where:

$$
\text{spillback}_r
$$

is the number of vehicles that exceed the storage capacity of ramp \(r\).

---

### Total Capacity Penalty

The total capacity penalty at one CTM time step is:

$$
L_{\text{cap}}
=
\sum_i L_{\text{doorway}, i}
+
\sum_i L_{\text{safe}, i}
+
\sum_i L_{\text{physical}, i}
+
\sum_r L_{\text{spillback}, r}
$$

This penalty is calculated at every 30-second CTM step and then summed over the full 120-step benchmark horizon.

The old synthetic 6-segment capacity test is not used in the final benchmark. The official benchmark uses the actual CTM states, flows, ramp queues, and spillback values generated during the 8-cell state-based simulation.

In [78]:
# 10. Capacity penalty function for 8-cell / 30-sec CTM
# Purpose: Compute the capacity-related penalties used in the official state-based benchmark, ADMM, and ADMM-MPC.

# Penalty weight
lambda_1 = 1.0   # doorway capacity penalty weight
lambda_2 = 0.5   # safe threshold penalty weight
lambda_3 = 1.0   # physical capacity penalty weight
lambda_4 = 0.5   # spillback penalty weight


def capacity_penalty_one_step(
    q_in,
    u_in_step,
    x_next,
    doorway_capacity,
    safe_threshold_capacity,
    physical_capacity,
    spillback_by_ramp,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4
):
    doorway_penalty_by_cell = {}
    safe_threshold_penalty_by_cell = {}
    physical_capacity_penalty_by_cell = {}
    spillback_penalty_by_ramp = {}

    total_doorway_penalty = 0.0
    total_safe_threshold_penalty = 0.0
    total_physical_capacity_penalty = 0.0
    total_spillback_penalty = 0.0



    # 1.  doorway, safe threshold, and physical capacity penalties
    for cell in x_next:
        # Doorway capacity penalty
        doorway_overflow = max(
            q_in[cell] + u_in_step[cell] - doorway_capacity[cell],
            0.0
        )
        doorway_penalty = lambda_1 * (doorway_overflow ** 2)

        doorway_penalty_by_cell[cell] = doorway_penalty
        total_doorway_penalty += doorway_penalty


        # Safe threshold penalty
        safe_overflow = max(
            x_next[cell] - safe_threshold_capacity[cell],
            0.0
        )

        safe_threshold_penalty = lambda_2 * (safe_overflow ** 2)

        safe_threshold_penalty_by_cell[cell] = safe_threshold_penalty
        total_safe_threshold_penalty += safe_threshold_penalty


        # Physical capacity penalty
        physical_overflow = max(
            x_next[cell] - physical_capacity[cell],
            0.0
        )

        physical_capacity_penalty = lambda_3 * (physical_overflow ** 2)

        physical_capacity_penalty_by_cell[cell] = physical_capacity_penalty
        total_physical_capacity_penalty += physical_capacity_penalty


    # 2. Ramp spillback penalty
    for ramp in spillback_by_ramp:

        spillback_penalty = lambda_4 * (
            spillback_by_ramp[ramp] ** 2
        )

        spillback_penalty_by_ramp[ramp] = spillback_penalty
        total_spillback_penalty += spillback_penalty
  # 3. Total capacity penalt
    total_capacity_penalty = (
        total_doorway_penalty
        + total_safe_threshold_penalty
        + total_physical_capacity_penalty
        + total_spillback_penalty
    )
    return {
        "doorway_penalty_by_cell": doorway_penalty_by_cell,
        "safe_threshold_penalty_by_cell": safe_threshold_penalty_by_cell,
        "physical_capacity_penalty_by_cell": physical_capacity_penalty_by_cell,
        "spillback_penalty_by_ramp": spillback_penalty_by_ramp,

        "total_doorway_penalty": total_doorway_penalty,
        "total_safe_threshold_penalty": total_safe_threshold_penalty,
        "total_physical_capacity_penalty": total_physical_capacity_penalty,
        "total_spillback_penalty": total_spillback_penalty,
        "total_capacity_penalty": total_capacity_penalty,
    }


In [79]:
# 11. CTM 30-second state update function
# Purpose: Update the 8-cell mainline state using CTM sending/receiving logic.


def ctm_30sec_step(
    x_current,
    q_in_boundary_step,
    u_in_step,
    f_out_step,
    doorway_capacity,
    physical_capacity
):
    cells = list(x_current.keys())

    sending = {}
    receiving = {}
    q_out = {}
    q_in = {}


    # 1. Sending flow from each cell
    # A cell cannot send more vehicles than it contains, and cannot exceed its doorway capacity.

    for cell in cells:
        sending[cell] = min(
            x_current[cell],
            doorway_capacity[cell]
        )


    # 2. Receiving capacity for each cell
    # A downstream cell can receive vehicles only if it has space.
    for cell in cells:
        receiving[cell] = max(
            physical_capacity[cell] - x_current[cell],
            0.0
        )


    # 3. Mainline outflow from each cell
    # For Cell 1–7:
    #     q_out is limited by upstream sending and downstream receiving.
    #
    # For Cell 8:
    #     q_out exits the corridor.
    for i in range(len(cells)):
        current_cell = cells[i]
        if i < len(cells) - 1:
            downstream_cell = cells[i + 1]

            q_out[current_cell] = min(
                sending[current_cell],
                receiving[downstream_cell]
            )

        else:
            q_out[current_cell] = sending[current_cell]


    # 4. Mainline inflow into each cell
    # Cell 1 receives boundary inflow from PeMS.
    # Cell 2–8 receive q_out from the upstream cell.

    for i in range(len(cells)):
        current_cell = cells[i]
        if i == 0:
            q_in[current_cell] = q_in_boundary_step
        else:
            upstream_cell = cells[i - 1]
            q_in[current_cell] = q_out[upstream_cell]


    # 5. CTM state update
    # x_next = x_current + q_in + ramp_in - q_out - off_ramp_out

    x_next = {}
    for cell in cells:
        x_next[cell] = (
            x_current[cell]
            + q_in[cell]
            + u_in_step[cell]
            - q_out[cell]
            - f_out_step[cell]
        )

        # Numerical safety: vehicle count cannot be negative
        x_next[cell] = max(x_next[cell], 0.0)


    return x_next, q_out, q_in, sending, receiving

In [80]:
# CTM first-step state update check
# Purpose: Verify that ctm_30sec_step runs using the first 30-second input.

ctm_test_x_current = {
    "Cell 1": 0.0,
    "Cell 2": 0.0,
    "Cell 3": 0.0,
    "Cell 4": 0.0,
    "Cell 5": 0.0,
    "Cell 6": 0.0,
    "Cell 7": 0.0,
    "Cell 8": 0.0,
}


ctm_test_observed_release_step = {
    "u1": observed_release_series["u1"][0],
    "u2": observed_release_series["u2"][0],
    "u3": observed_release_series["u3"][0],
    "u4": observed_release_series["u4"][0],
    "u5": observed_release_series["u5"][0],
}


ctm_test_u_in_step = {
    "Cell 1": 0.0,
    "Cell 2": ctm_test_observed_release_step["u1"],
    "Cell 3": 0.0,
    "Cell 4": ctm_test_observed_release_step["u2"],
    "Cell 5": ctm_test_observed_release_step["u3"],
    "Cell 6": ctm_test_observed_release_step["u4"],
    "Cell 7": ctm_test_observed_release_step["u5"],
    "Cell 8": 0.0,
}


ctm_test_x_next, ctm_test_q_out, ctm_test_q_in, ctm_test_sending, ctm_test_receiving = ctm_30sec_step(
    ctm_test_x_current,
    q_in_boundary_series[0],
    ctm_test_u_in_step,
    f_out_series[0],
    doorway_capacity,
    physical_capacity
)


ctm_first_step_check_df = pd.DataFrame({
    "cell": list(ctm_test_x_next.keys()),
    "x_current": [
        ctm_test_x_current[cell]
        for cell in ctm_test_x_next
    ],
    "q_in": [
        ctm_test_q_in[cell]
        for cell in ctm_test_x_next
    ],
    "u_in": [
        ctm_test_u_in_step[cell]
        for cell in ctm_test_x_next
    ],
    "q_out": [
        ctm_test_q_out[cell]
        for cell in ctm_test_x_next
    ],
    "f_out": [
        f_out_series[0][cell]
        for cell in ctm_test_x_next
    ],
    "x_next": [
        ctm_test_x_next[cell]
        for cell in ctm_test_x_next
    ],
})


print(" CTM First-Step State Update Check ")
display(ctm_first_step_check_df.round(3))

 CTM First-Step State Update Check 


,cell,x_current,q_in,u_in,q_out,f_out,x_next
0,Cell 1,0.0,64.5,0.0,0.0,0.0,64.5
1,Cell 2,0.0,0.0,8.8,0.0,0.0,8.8
2,Cell 3,0.0,0.0,0.0,0.0,0.0,0.0
3,Cell 4,0.0,0.0,3.9,0.0,0.0,3.9
4,Cell 5,0.0,0.0,7.3,0.0,0.0,7.3
5,Cell 6,0.0,0.0,5.6,0.0,8.2,0.0
6,Cell 7,0.0,0.0,6.4,0.0,0.8,5.6
7,Cell 8,0.0,0.0,0.0,0.0,0.0,0.0


## 12–14. Official 8-Cell CTM State-Based Mainline Delay Setup

This section prepares the official **state-based CTM benchmark** used for comparison with ADMM and ADMM-MPC.

Unlike the earlier flow/speed-based sanity check, this section uses the actual CTM state variables:

$$
x_i(t)
$$

where \(x_i(t)\) is the number of vehicles stored in CTM Cell \(i\) at time step \(t\).

The benchmark uses:

$$
8 \text{ CTM cells}
$$

$$
\Delta t = 0.5 \text{ minutes}
$$

$$
120 \text{ steps}
$$

which corresponds to:

$$
08{:}00 \text{ to } 09{:}00
$$

---

### 12. Real 8-Cell Initial State

The initial state represents the number of vehicles in each CTM cell at the beginning of the benchmark window.

That is:

$$
x_i(0)
$$

for each cell:

$$
i = 1, 2, \dots, 8
$$

These values are used as the starting point for the CTM simulation.

The ramp queues are also initialized before the simulation begins. In this benchmark, the initial ramp queues are set to zero:

$$
R_j(0) = 0
$$

for each ramp:

$$
j = 1, 2, \dots, 5
$$

This means the no-control benchmark starts without any pre-existing ramp queue.

---

### 13. 8-Cell Free-Flow Travel Time Calculation

The CTM delay formula requires a free-flow travel time for each CTM cell:

$$
TT_{\text{ff}, i}
$$

Instead of hard-coding these values, they are calculated from:

- mainline detector postmiles,
- the 6 detector-to-detector physical segment speeds,
- and the 8 equal CTM cells.

Because the CTM uses 8 equal cells but the detector spacing creates 6 physical road segments, each CTM cell may overlap with one or more physical segments.

For each CTM cell, the free-flow travel time is calculated as:

$$
TT_{\text{ff}, i}
=
\sum_s
\left(
\frac{L_{i,s}}{v_{\text{ff},s}}
\right)
\cdot 60
$$

where:

$$
L_{i,s}
$$

is the overlap length between CTM Cell \(i\) and physical segment \(s\),

$$
v_{\text{ff},s}
$$

is the free-flow speed of physical segment \(s\),

and the factor \(60\) converts hours into minutes.

This produces one free-flow travel time value for each CTM cell:

$$
TT_{\text{ff},1}, TT_{\text{ff},2}, \dots, TT_{\text{ff},8}
$$

---

### 14. State-Based Mainline Delay Calculation

The mainline delay is calculated from the CTM states, not directly from observed speed.

For each CTM cell, the total travel time spent in the cell during one 30-second step is:

$$
TTT_i(t)
=
\frac{x_i(t) + x_i(t+1)}{2}
\cdot
\Delta t
$$

where:

$$
x_i(t)
$$

is the number of vehicles in Cell \(i\) before the CTM update,

$$
x_i(t+1)
$$

is the number of vehicles in Cell \(i\) after the CTM update,

and:

$$
\Delta t = 0.5 \text{ minutes}
$$

The free-flow component is:

$$
(q_{\text{out},i}(t) + f_i(t)) \cdot TT_{\text{ff},i}
$$

where:

$$
q_{\text{out},i}(t)
$$

is the mainline outflow from Cell \(i\),

$$
f_i(t)
$$

is the off-ramp flow leaving Cell \(i\),

and:

$$
TT_{\text{ff},i}
$$

is the free-flow travel time for Cell \(i\).

Therefore, the state-based mainline delay for Cell \(i\) is:

$$
D_{M,i}(t)
=
TTT_i(t)
-
(q_{\text{out},i}(t) + f_i(t)) \cdot TT_{\text{ff},i}
$$

The total mainline delay for one CTM step is:

$$
D_M(t)
=
\sum_{i=1}^{8}
D_{M,i}(t)
$$

This function is later applied over all 120 CTM steps to compute the official state-based benchmark mainline delay.

In [84]:
# 12. Final 8-cell CTM state setup
# Purpose: Define the real initial mainline state and ramp queue state for the official state-based CTM benchmark.

#08:00 mainline initial state
# Units: vehicles in each CTM cell

mainline_initial_state = {
    "Cell 1": 122.729,
    "Cell 2": 118.944,
    "Cell 3": 95.457,
    "Cell 4": 119.602,
    "Cell 5": 141.365,
    "Cell 6": 190.091,
    "Cell 7": 185.566,
    "Cell 8": 172.041,
}


# Initial ramp queue
# Benchmark starts with zero queue unless you intentionally
# choose a nonzero fill ratio.
 
ramp_queue_0 = {
    "u1": 0.0,
    "u2": 0.0,
    "u3": 0.0,
    "u4": 0.0,
    "u5": 0.0,
}


# Replace ctm_30sec_step with ADMM-matching version
def ctm_30sec_step(
    x_current,
    q_in_boundary_step,
    u_in_step,
    f_out_step,
    doorway_capacity,
    physical_capacity
):
    sending = {}
    receiving = {}
    q_out = {}
    q_in = {}
    x_next = {}

    cells = [f"Cell {i}" for i in range(1, 9)]


    # 1. Sending flow
    # A cell cannot send more vehicles than it contains,
    # and cannot exceed its doorway capacity.
    for cell in cells:
        sending[cell] = min(
            x_current[cell],
            doorway_capacity[cell]
        )


    # 2. Receiving flow
    # A downstream cell cannot receive more than:
    #   - its remaining physical space
    #   - its doorway capacity
    for cell in cells:
        receiving[cell] = max(
            0.0,
            min(
                doorway_capacity[cell],
                physical_capacity[cell] - x_current[cell]
            )
        )


    # 3. Mainline outflow
    # Cell 1–7 outflow is limited by downstream receiving.
    # Cell 8 exits the corridor.
    for i in range(len(cells)):

        current_cell = cells[i]

        if i < len(cells) - 1:
            downstream_cell = cells[i + 1]

            q_out[current_cell] = min(
                sending[current_cell],
                receiving[downstream_cell]
            )

        else:
            q_out[current_cell] = sending[current_cell]


    # 4. Mainline inflow
    # Cell 1 receives boundary inflow.
    # Other cells receive upstream outflow.
    q_in["Cell 1"] = q_in_boundary_step

    for i in range(1, len(cells)):
        current_cell = cells[i]
        upstream_cell = cells[i - 1]

        q_in[current_cell] = q_out[upstream_cell]


    # 5. CTM state update
    # x_next = x_current + q_in + u_in - q_out - f_out
    for cell in cells:
        x_next[cell] = (
            x_current[cell]
            + q_in[cell]
            + u_in_step[cell]
            - q_out[cell]
            - f_out_step[cell]
        )

        x_next[cell] = max(x_next[cell], 0.0)


    return x_next, q_out, q_in, sending, receiving


# Clean check

mainline_initial_state_df = pd.DataFrame({
    "cell": list(mainline_initial_state.keys()),
    "initial_state_x0": list(mainline_initial_state.values()),
    "safe_threshold_capacity": [
        safe_threshold_capacity[cell]
        for cell in mainline_initial_state
    ],
    "physical_capacity": [
        physical_capacity[cell]
        for cell in mainline_initial_state
    ],
})

ramp_queue_0_df = pd.DataFrame({
    "ramp": list(ramp_queue_0.keys()),
    "initial_queue": list(ramp_queue_0.values()),
    "max_queue": [
        ramp_max_queue_by_u[ramp]
        for ramp in ramp_queue_0
    ],
})

print("8-Cell Mainline Initial State")
display(mainline_initial_state_df.round(3))

print("Initial Ramp Queue ")
display(ramp_queue_0_df.round(3))

8-Cell Mainline Initial State


,cell,initial_state_x0,safe_threshold_capacity,physical_capacity
0,Cell 1,122.729,413.744,591.062
1,Cell 2,118.944,413.744,591.062
2,Cell 3,95.457,413.744,591.062
3,Cell 4,119.602,413.744,591.062
4,Cell 5,141.365,496.492,709.275
5,Cell 6,190.091,413.744,591.062
6,Cell 7,185.566,413.744,591.062
7,Cell 8,172.041,413.744,591.062


Initial Ramp Queue 


,ramp,initial_queue,max_queue
0,u1,0.0,28.669
1,u2,0.0,72.356
2,u3,0.0,56.168
3,u4,0.0,72.441
4,u5,0.0,48.819


In [91]:
# 13. Calculate 8-cell free-flow travel time
# Purpose: Convert the 6 detector-to-detector free-flow segments into 8 equal CTM-cell free-flow travel times.

# Mainline detector postmiles
mainline_station_postmile = (
    selected_metadata_clean[
        selected_metadata_clean["station_id"].isin(mainline_ids)
    ][
        ["station_id", "station_name", "absolute_postmile"]
    ]
    .sort_values("absolute_postmile")
    .reset_index(drop=True)
)


# Physical detector-to-detector segment free-flow speeds

physical_segment_speed_df = pd.DataFrame({
    "physical_segment": [
        "RED HILL → BRISTOL 1",
        "BRISTOL 1 → FAIRVIEW",
        "FAIRVIEW → HARBOR 1",
        "HARBOR 1 → HARBOR 2",
        "HARBOR 2 → EUCLID",
        "EUCLID → TALBERT",
    ],
    "start_postmile": [
        mainline_station_postmile.loc[0, "absolute_postmile"],
        mainline_station_postmile.loc[1, "absolute_postmile"],
        mainline_station_postmile.loc[2, "absolute_postmile"],
        mainline_station_postmile.loc[3, "absolute_postmile"],
        mainline_station_postmile.loc[4, "absolute_postmile"],
        mainline_station_postmile.loc[5, "absolute_postmile"],
    ],
    "end_postmile": [
        mainline_station_postmile.loc[1, "absolute_postmile"],
        mainline_station_postmile.loc[2, "absolute_postmile"],
        mainline_station_postmile.loc[3, "absolute_postmile"],
        mainline_station_postmile.loc[4, "absolute_postmile"],
        mainline_station_postmile.loc[5, "absolute_postmile"],
        mainline_station_postmile.loc[6, "absolute_postmile"],
    ],
    "v_ff_mph": [
        v_ff_1,
        v_ff_2,
        v_ff_3,
        v_ff_4,
        v_ff_5,
        v_ff_6,
    ],
})


# Build 8 equal CTM cells over full corridor

corridor_start_postmile = mainline_station_postmile["absolute_postmile"].min()
corridor_end_postmile = mainline_station_postmile["absolute_postmile"].max()

num_cells = 8
cell_length = (corridor_end_postmile - corridor_start_postmile) / num_cells


ctm_cell_df = pd.DataFrame({
    "cell": [f"Cell {i}" for i in range(1, num_cells + 1)],
    "cell_start_postmile": [
        corridor_start_postmile + (i - 1) * cell_length
        for i in range(1, num_cells + 1)
    ],
    "cell_end_postmile": [
        corridor_start_postmile + i * cell_length
        for i in range(1, num_cells + 1)
    ],
})


# Helper: overlap length between one CTM cell and one physical segment

def overlap_length(cell_start, cell_end, segment_start, segment_end):
    overlap = max(
        0.0,
        min(cell_end, segment_end) - max(cell_start, segment_start)
    )

    return overlap


# Calculate TT_ff for each CTM cell

tt_ff_min = {}
tt_ff_rows = []

for _, cell_row in ctm_cell_df.iterrows():

    cell = cell_row["cell"]
    cell_start = cell_row["cell_start_postmile"]
    cell_end = cell_row["cell_end_postmile"]

    cell_tt_ff_min = 0.0

    for _, segment_row in physical_segment_speed_df.iterrows():

        segment_start = segment_row["start_postmile"]
        segment_end = segment_row["end_postmile"]
        segment_speed = segment_row["v_ff_mph"]

        overlap = overlap_length(
            cell_start,
            cell_end,
            segment_start,
            segment_end
        )

        if overlap > 0:
            overlap_tt_min = (overlap / segment_speed) * 60.0
            cell_tt_ff_min += overlap_tt_min

    tt_ff_min[cell] = cell_tt_ff_min

    tt_ff_rows.append({
        "cell": cell,
        "cell_start_postmile": cell_start,
        "cell_end_postmile": cell_end,
        "cell_length_miles": cell_end - cell_start,
        "TT_ff_min": cell_tt_ff_min,
    })


tt_ff_min_df = pd.DataFrame(tt_ff_rows)


print("Calculated 8-Cell Free-Flow Travel Time ")
display(tt_ff_min_df.round(3))

print("Calculated tt_ff_min dictionary:")
for cell in tt_ff_min:
    print(cell, ":", round(tt_ff_min[cell], 3))


Calculated 8-Cell Free-Flow Travel Time 


,cell,cell_start_postmile,cell_end_postmile,cell_length_miles,TT_ff_min
0,Cell 1,8.170,8.782,0.613,0.567
1,Cell 2,8.782,9.395,0.612,0.564
2,Cell 3,9.395,10.008,0.613,0.544
3,Cell 4,10.008,10.620,0.613,0.533
4,Cell 5,10.620,11.232,0.612,0.534
5,Cell 6,11.232,11.845,0.613,0.537
6,Cell 7,11.845,12.458,0.612,0.538
7,Cell 8,12.458,13.070,0.613,0.540


Calculated tt_ff_min dictionary:
Cell 1 : 0.567
Cell 2 : 0.564
Cell 3 : 0.544
Cell 4 : 0.533
Cell 5 : 0.534
Cell 6 : 0.537
Cell 7 : 0.538
Cell 8 : 0.54


In [90]:
# 14. Mainline delay setup for 8-cell CTM
# Purpose: Define the free-flow travel time for each CTM cell and calculate state-based mainline delay for one 30-second CTM step.

delta_t = 0.5  # 30 seconds = 0.5 minutes

# Build CTM ramp inflow dictionary from observed ramp releases
def build_u_in_from_observed_release(observed_release_step):
    u_in_step = {
        "Cell 1": 0.0,
        "Cell 2": observed_release_step["u1"],  # Bristol 1
        "Cell 3": 0.0,
        "Cell 4": observed_release_step["u2"],  # Fairview
        "Cell 5": observed_release_step["u3"],  # Harbor 1
        "Cell 6": observed_release_step["u4"],  # Harbor 2
        "Cell 7": observed_release_step["u5"],  # Euclid
        "Cell 8": 0.0,
    }

    return u_in_step


# Mainline delay for one 30-second CTM step

def mainline_delay_one_step(
    x_now,
    x_next,
    q_out,
    f_out,
    tt_ff_min,
    delta_t
):
    rows = []
    total_mainline_delay = 0.0

    for cell in x_now:

        # Actual total time spent in cell during this CTM step
        ttt = ((x_now[cell] + x_next[cell]) / 2.0) * delta_t

        # Free-flow time for vehicles leaving through mainline/out-ramp
        ff_term = (q_out[cell] + f_out[cell]) * tt_ff_min[cell]

        # State-based mainline delay
        delay = ttt - ff_term

        total_mainline_delay += delay

        rows.append({
            "Cell": cell,
            "x_now": x_now[cell],
            "x_next": x_next[cell],
            "q_out": q_out[cell],
            "f_out": f_out[cell],
            "TTT_veh_min": ttt,
            "free_flow_component": ff_term,
            "mainline_delay_veh_min": delay,
        })

    mainline_delay_df = pd.DataFrame(rows)

    return mainline_delay_df, total_mainline_delay


# first-step check using actual 08:00 initial state
observed_release_step_0 = {
    "u1": observed_release_series["u1"][0],
    "u2": observed_release_series["u2"][0],
    "u3": observed_release_series["u3"][0],
    "u4": observed_release_series["u4"][0],
    "u5": observed_release_series["u5"][0],
}

u_in_step_0 = build_u_in_from_observed_release(
    observed_release_step_0
)

f_out_step_0 = f_out_series[0]

x_next_0, q_out_0, q_in_0, sending_0, receiving_0 = ctm_30sec_step(
    mainline_initial_state,
    q_in_boundary_series[0],
    u_in_step_0,
    f_out_step_0,
    doorway_capacity,
    physical_capacity
)

mainline_delay_df_0, total_mainline_delay_0 = mainline_delay_one_step(
    mainline_initial_state,
    x_next_0,
    q_out_0,
    f_out_step_0,
    tt_ff_min,
    delta_t
)

print(" First-Step CTM Mainline Delay Check ")
display(mainline_delay_df_0.round(3))

print("Total first-step mainline delay:", round(total_mainline_delay_0, 3), "veh-min")

 First-Step CTM Mainline Delay Check 


,Cell,x_now,x_next,q_out,f_out,TTT_veh_min,free_flow_component,mainline_delay_veh_min
0,Cell 1,122.729,99.819,87.41,0.0,55.637,49.596,6.041
1,Cell 2,118.944,127.744,87.41,0.0,61.672,49.315,12.357
2,Cell 3,95.457,95.457,87.41,0.0,47.728,47.572,0.156
3,Cell 4,119.602,123.502,87.41,0.0,60.776,46.552,14.224
4,Cell 5,141.365,148.665,87.41,0.0,72.508,46.641,25.866
5,Cell 6,190.091,187.491,87.41,8.2,94.396,51.381,43.014
6,Cell 7,185.566,191.166,87.41,0.8,94.183,47.492,46.691
7,Cell 8,172.041,172.041,87.41,0.0,86.020,47.240,38.781


Total first-step mainline delay: 187.131 veh-min


## Safe-threshold penalty calculation
## we used the same syntatic data for final state of the segment (X_final) to varify if the calculation works

In [ ]:
# Safe-threshold penalty calculation

lambda_2_values = [0, 0.5, 1, 5]

def safe_threshold_penalty(lambda_2, x_final, x_safe):
    overflow = max(0, x_final - x_safe)
    penalty = lambda_2 * (overflow ** 2)
    return overflow, penalty

for eta_label in safe_occupancy:
    print("\n" + eta_label)

    for lambda_2 in lambda_2_values:
        print("\nlambda_2 =", lambda_2)
        total_safe_penalty = 0

        for segment in X_final:
            overflow, penalty = safe_threshold_penalty(
                lambda_2,
                X_final[segment],
                safe_occupancy[eta_label][segment]
            )

            total_safe_penalty += penalty

            print(
                segment,
                "X_final =", round(X_final[segment], 3),
                ", X_safe =", round(safe_occupancy[eta_label][segment], 3),
                ", overflow =", round(overflow, 3),
                ", penalty =", round(penalty, 3)
            )

        print("Total safe threshold penalty =", round(total_safe_penalty, 3))

In [ ]:
## safe threshold penalty with synthetic X_final

X_final_fake = {
    "Segment 1": N_max_value["Segment 1"] + 50,
    "Segment 2": N_max_value["Segment 2"] + 60,
    "Segment 3": N_max_value["Segment 3"] + 70,
    "Segment 4": N_max_value["Segment 4"] + 80,
    "Segment 5": N_max_value["Segment 5"] + 90,
    "Segment 6": N_max_value["Segment 6"] + 100
}

lambda_2_values = [0, 0.5, 1, 5]

def safe_threshold_penalty(lambda_2, X_final_fake, x_safe):
    overflow = max(0, X_final_fake - x_safe)
    penalty = lambda_2 * (overflow ** 2)
    return overflow, penalty

for eta_label in safe_occupancy:
    print("\n" + eta_label)

    for lambda_2 in lambda_2_values:
        print("\nlambda_2 =", lambda_2)
        total_safe_penalty = 0

        for segment in X_final_fake:
            overflow, penalty = safe_threshold_penalty(
                lambda_2,
                X_final_fake[segment],
                safe_occupancy[eta_label][segment]
            )

            total_safe_penalty += penalty

            print(
                segment,
                "X_final_fake =", round(X_final_fake[segment], 3),
                ", X_safe =", round(safe_occupancy[eta_label][segment], 3),
                ", overflow =", round(overflow, 3),
                ", penalty =", round(penalty, 3)
            )

        print("Total safe threshold penalty =", round(total_safe_penalty, 3))

# Above calculations are all done with the dataset given from PeMS

